# Week 08: Building the AI/ML pipeline

Last week you built every diffusion-specific piece of mathematics the model
needs — in pure numpy, with every line transparent and debuggable. The
**forward process** that maps a clean residual to noise across T = 200 timesteps,
the **cosine noise schedule** that defines α_t and σ_t at every t, and the
empirical verifications that the corruption behaves the way the math says it
should. The window table, the stratified split, and the schedule are all
saved to `diffusion_windows.parquet`. What you have *not* built is anything
with learnable parameters.

That changes this week. The forward apparatus from Week 07 will be reused
unchanged; on top of it we add the **reverse process** — a small denoising
neural network whose job is to estimate ε from r_t, trained in PyTorch with
PyTorch Lightning. By the end of this notebook you will have an end-to-end
**unconditional diffusion model** that learns the marginal distribution of
residuals, samples from it, and — through a deliberate architectural
ablation — answers a real empirical question: does the network's awareness
of the noise level (the **timestep embedding**) actually earn its keep on
this dataset?

**Strategic context.** This week is the most code-heavy of the three. The
conceptual lift is smaller than Week 07's diffusion math — the hard ideas
(forward process, schedule, ε-prediction) are already in your toolkit. What
this week demands is *integration*: turning the numpy schedule into a
PyTorch buffer, turning the residual table into a Dataset, turning the
training equation into a Lightning training step, and watching the whole
thing learn. Most of you will encounter the canonical AI/ML pipeline
(Dataset → DataLoader → Model → LightningModule → Trainer) for the first
time here. The pipeline is not diffusion-specific — every PyTorch project
you ever build will have the same five pieces — but each piece has a
diffusion-specific wrinkle, and we will flag those wrinkles as we go.

**One deliberate omission this week: conditioning.** The Week 09 model will
condition on (cycle amplitude, universal-path latitude) and run through
`compute_global_nll`. This week we train the *unconditional* version: it
learns the marginal distribution of residuals across all training cycles
and can sample plausible residuals, but it cannot target a specific window.
The reason for the split is pedagogical: build the diffusion machinery
first, feel it work end-to-end, and only then add the conditioning layer
on top. One new idea per week.

**By the end of this notebook you should be able to:**
- Wrap the Week 07 residual table (loaded from `diffusion_windows.parquet`)
  in a **PyTorch Dataset** and explain why the dataset for diffusion training
  returns clean residuals only — without any noise injection or timestep
  sampling.
- Build the three **DataLoaders** (train / val / test) using the `split`
  column already present in the parquet, and articulate why shuffling
  matters for training but not for validation.
- Implement a **sinusoidal timestep embedding** that maps the integer t to a
  dense vector, and visualize how the embedding rotates through embedding
  space as t varies.
- Build a **DiffusionMLP** that takes (r_t, t) and returns ε̂, with a
  constructor flag that toggles whether the timestep embedding is used at
  all — the architectural ablation that lets you measure whether t-awareness
  earns its keep.
- Write a **PyTorch Lightning training loop** that samples t per batch
  element, draws ε, applies the Week 07 forward equation in PyTorch, and
  trains on MSE loss against the true ε.
- Implement a **DDIM-style sampler** that runs the trained model in reverse
  to generate new residuals, and verify that the sampled residuals match
  the training distribution on bin-wise mean, standard deviation, and
  covariance structure.
- Run the with-vs-without-timestep-embedding ablation and read its result
  honestly, including the case where it tells you something uncomfortable
  about your architecture choice.


In [ ]:
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    subprocess.run(["git", "-C", repo_path, "pull"], check=True)
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
import os, sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

In [ ]:
# ── locate repo root robustly (same scheme as Week 07) ──────────────────────
_cwd = os.getcwd()
_week8_dir = _repo_root = None
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(1, 5)]:
    if os.path.isfile(os.path.join(_base, "weeks", "week_08", "butterflAI_model.py")):
        _week8_dir = os.path.join(_base, "weeks", "week_08")
        _repo_root = _base
        break
    if os.path.isfile(os.path.join(_base, "butterflAI_model.py")) and "week_08" in _base:
        _week8_dir = _base
        _repo_root = os.path.abspath(os.path.join(_base, "../.."))
        break

if _week8_dir is None:
    raise FileNotFoundError(
        "Cannot locate butterflAI_model.py. Make sure repo_path points to the "
        "butterflai repo root, and that you have run Week 07 to produce "
        "diffusion_windows.parquet."
    )
for _p in [_week8_dir, _repo_root]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── load Week 07 outputs ─────────────────────────────────────────────────────
windows_df = pd.read_parquet(Path(_week8_dir) / "diffusion_windows.parquet")

hist_cols = [f"hist_emp_{j:02d}" for j in range(15)]
par_cols  = [f"hist_par_{j:02d}" for j in range(15)]

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

# ── load the official ButterflAI classical model ─────────────────────────────
from butterflAI_model import ButterflAIModel
classical = ButterflAIModel(os.path.join(_week8_dir, "official_model.npz"))

print(f"Loaded {len(windows_df)} windows from diffusion_windows.parquet")
print(f"  splits : {windows_df['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles : {sorted(windows_df['cycle'].unique())}")
print(classical)

In [ ]:
# ── recompute the Week 07 cosine schedule (numpy version) ──────────────────
# Same formula as Week 07 Task 28. Kept here so this notebook is
# self-contained — the LightningModule in Task 40 will build a torch
# version of the same schedule from these arrays.
T, s_offset = 200, 0.008
_t_arr         = np.arange(T + 1, dtype=float)
_f0            = np.cos(np.pi / 2 * s_offset / (1 + s_offset)) ** 2
_alpha_bar_raw = np.cos(np.pi / 2 * (_t_arr / T + s_offset) / (1 + s_offset)) ** 2 / _f0
_alpha_bar_raw = np.clip(_alpha_bar_raw, 0.0, 1.0)
_beta = np.zeros(T + 1)
for t in range(1, T + 1):
    _beta[t] = np.clip(1.0 - _alpha_bar_raw[t] / _alpha_bar_raw[t - 1], 1e-8, 0.999)
alpha_bar_np = np.ones(T + 1)
for t in range(1, T + 1):
    alpha_bar_np[t] = alpha_bar_np[t - 1] * (1.0 - _beta[t])
alpha_np = np.sqrt(alpha_bar_np)
sigma_np = np.sqrt(np.clip(1.0 - alpha_bar_np, 0.0, 1.0))

print(f"T={T}, schedule arrays length {len(alpha_np)}")

In [ ]:
# ── Inspect the classical model on any cycle in windows_df ─────────────────
# Change cycle_number to see a different cycle.  Both hemispheres are shown
# simultaneously.  Filled profiles = empirical 6-month histogram (hist_emp_*);
# dashed curves = classical ButterflAI Gaussian evaluated at the same (A, τ).
cycle_number  = 24      # ← any value in sorted(windows_df['cycle'].unique())
PROFILE_SCALE = 0.40    # max density → this many τ-years of horizontal width

# ── select & sort windows ─────────────────────────────────────────────────
wdf_n = (windows_df[(windows_df["cycle"] == cycle_number) &
                     (windows_df["hemisphere"] == "north")]
         .sort_values("tau_center").reset_index(drop=True))
wdf_s = (windows_df[(windows_df["cycle"] == cycle_number) &
                     (windows_df["hemisphere"] == "south")]
         .sort_values("tau_center").reset_index(drop=True))

if wdf_n.empty and wdf_s.empty:
    raise ValueError(
        f"Cycle {cycle_number} not found. "
        f"Available: {sorted(windows_df['cycle'].unique())}"
    )

# Shared normalization across both hemispheres so relative amplitudes are comparable
_cyc_mask  = windows_df["cycle"] == cycle_number
global_max = max(windows_df[_cyc_mask][hist_cols].values.max(), 1e-9)
scale      = PROFILE_SCALE / global_max

# Shared τ range for colormap — same colour = same time in both panels
_all_tau   = pd.concat([wdf_n["tau_center"], wdf_s["tau_center"]])
tau_norm   = plt.Normalize(vmin=_all_tau.min(), vmax=_all_tau.max())
cmap_win   = plt.get_cmap("viridis")

# Step-function bin edges for fill_betweenx  (15 bins → 30 edge positions)
_lat_edges_step = np.concatenate(
    [[LAT_BINS[0]], np.repeat(LAT_BINS[1:-1], 2), [LAT_BINS[-1]]]
)   # shape (30,)
_lat_fine = np.linspace(0, 45, 300)   # fine grid for smooth Gaussian curves

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.subplots_adjust(hspace=0.06)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

for ax, wdf, sign, hemi_label in [
    (axes[0], wdf_n, +1, "North"),
    (axes[1], wdf_s, -1, "South"),
]:
    if wdf.empty:
        ax.text(0.5, 0.5, f"No data for {hemi_label}",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=13, color="gray")
        ax.set_ylabel("Latitude (°)")
        continue

    for _, row in wdf.iterrows():
        tau   = row["tau_center"]
        A     = row["amplitude"]
        color = cmap_win(tau_norm(tau))

        # ── empirical histogram: filled step-function profile ────────────
        hist_vals = row[hist_cols].values.astype(float)
        hist_step = np.repeat(hist_vals, 2)        # (30,)
        x_hist    = tau + hist_step * scale

        lats_plot = sign * _lat_edges_step
        ax.fill_betweenx(lats_plot, tau, x_hist,
                         color=color, alpha=0.35, linewidth=0, zorder=2)
        ax.plot(x_hist, lats_plot,
                color=color, linewidth=0.8, alpha=0.75, zorder=3)

        # ── classical Gaussian: smooth dashed curve ───────────────────────
        gauss_vals = classical.density(A, tau, _lat_fine)
        x_gauss    = tau + gauss_vals * scale
        ax.plot(x_gauss, sign * _lat_fine,
                color=color, linewidth=2.0, linestyle="--", alpha=0.9, zorder=5)

    ax.set_ylabel(f"Latitude (°) — {hemi_label}", fontsize=10)
    ax.set_ylim((0, 48) if sign == 1 else (-48, 0))

    if sign == 1:   # legend once, in the north panel
        ax.legend(
            handles=[
                Patch(facecolor="gray", alpha=0.45,
                      label="Empirical histogram (hist_emp_*)"),
                Line2D([0], [0], color="gray", linewidth=2, linestyle="--",
                       label="Classical Gaussian — ButterflAI model"),
            ],
            loc="upper right", fontsize=9,
        )

# ── single shared colorbar spanning both panels ───────────────────────────
sm = plt.cm.ScalarMappable(cmap="viridis", norm=tau_norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=axes, pad=0.01, fraction=0.02)
cb.set_label("τ (yr)", fontsize=10)

axes[1].set_xlabel("τ  (years from reference epoch)", fontsize=11)
fig.suptitle(
    f"Cycle {cycle_number}  —  6-month windows: "
    "empirical histograms (filled) overplotted with classical ButterflAI Gaussians (dashed)\n"
    "Profiles share the same horizontal scale; colour = τ (same scale in both panels)",
    fontsize=11,
)
plt.show()

---
# Week 08 Tasks: Building the AI/ML pipeline

Everything above this line is **setup**: the Week 07 outputs are loaded,
the cosine schedule is recomputed in numpy for reference, and the
official `ButterflAIModel` is instantiated. Below this line begins the
Week 08 work proper: build the dataset, the model, the training loop, the
sampler, and the ablation comparison.

The tasks build sequentially. Tasks 33–35 wrap the Week 07 residual table
in a PyTorch Dataset and DataLoader. Tasks 36–39 build the model in three
layered pieces (sinusoidal embedding → embedding module → main network)
and then sanity-test it. Task 40 stitches the model and the schedule into
a LightningModule. Task 41 trains it. Tasks 42–43 implement sampling and
verify the sampled distribution matches the training distribution. Task 44
runs the ablation experiment and produces the comparison table that closes
the week.

A note on convention: throughout this notebook, all tensor shape
manipulation uses `einops.rearrange` and `einops.repeat` with named
dimensions rather than `view`, `reshape`, `squeeze`, `unsqueeze`,
`expand`, or `tile`. The named-dimension form makes the shape semantics
explicit, which is especially useful when multiple dimensions could
plausibly be the batch dimension. `einops` was already required for
Week 07 and is available in the environment.


---
## Task 33 — Wrap the residual table in a PyTorch Dataset

The Week 07 residual table has one row per (cycle, hemisphere, 6-month
window) and a `split` column tagging each row as `'train'`, `'val'`, or
`'test'`. Most diffusion training pipelines call this object a **Dataset**:
an indexed collection of clean data points that the training loop will
draw from at random.

The dataset for
diffusion training is simple. In supervised learning  `__getitem__` typically returns an (input, target) pair. For diffusion, that is
not the right factoring. The triple (r_t, t, ε) the network actually trains
on is generated *inside* the Lightning training step, not in `__getitem__`.
The dataset returns just the clean residual r; the timestep t is sampled
fresh per batch, the noise ε is drawn fresh per batch, and the corruption
r_t = α_t · r + σ_t · ε is computed on the fly using the same formula
your Week 07 `forward_corrupt` implemented. This separation is deliberate:
it lets you change the timestep sampling distribution, the ε distribution,
or the loss weighting without ever touching the dataset.

This week the dataset returns *only*
the residual, as a tensor of shape (15,). Week 09 will add conditioning on
(amplitude, mu_universal), which means the dataset will need to return
those covariates alongside the residual. We do not preemptively wire that
in this week — keeping the dataset minimal makes the diffusion-specific
machinery easier to see — but the parquet already contains those columns,
so the Week 09 refactor will be small.

**Tasks:**
- Define a class `ResidualDataset(torch.utils.data.Dataset)` that wraps
  `windows_df`.
- The constructor takes the full `windows_df` and a string `split` in
  `{'train', 'val', 'test'}`, and filters the DataFrame to retain only
  rows whose `split` column matches.
- `__len__` returns the number of rows after filtering.
- `__getitem__(idx)` extracts the residual (the difference between the 15
  `hist_emp_*` columns and the 15 `hist_par_*` columns) and returns it as
  a single `torch.float32` tensor of shape `(15,)`.

**Important:** the residual is the *difference* between the empirical and
parametric histograms (both densities). It is not normalized to unit
variance, not centered at zero, and not bounded — it is in the same units
as the histograms (probability density per latitude). The diffusion model
will learn to generate residuals with whatever distributional structure
the training data exhibits, so do not preprocess them here. The
`hist_par_*` columns themselves were generated in Week 07 by integrating
the official classical model's per-window Gaussian over each bin; the
residual you return here is the part of the per-window density the
classical model leaves on the table.


In [ ]:
# Put your code here for Task 33.
# Task 33: ResidualDataset
# Depends on: windows_df, emp_cols, par_cols (from setup)

import torch
from torch.utils.data import Dataset

class ResidualDataset(Dataset):
    """
    Wraps the Week 07 window table for diffusion training.

    Returns one clean residual tensor per index — no noise injection,
    no timestep sampling. Both happen inside the Lightning training step
    so that every draw of a given residual sees a fresh (t, ε) pair.

    Parameters
    ----------
    windows_df : pd.DataFrame
        Full window table from Task 26, with 'split' column.
    split : str
        One of 'train', 'val', 'test'. Only rows matching this tag
        are retained.

    __getitem__ returns
    -------------------
    r : torch.float32 tensor, shape (15,)
        hist_emp − hist_par for that window, in density units.
        Not normalized, not centered — raw residual as computed in Task 26.
    """

    # Column lists are class-level constants so they're computed once
    _EMP_COLS = [f"hist_emp_{k:02d}" for k in range(15)]
    _PAR_COLS = [f"hist_par_{k:02d}" for k in range(15)]

    def __init__(self, windows_df: pd.DataFrame, split: str):
        assert split in {"train", "val", "test"}, \
            f"split must be 'train', 'val', or 'test', got '{split}'"

        # Filter to the requested split
        mask = windows_df["split"] == split
        self._df = windows_df[mask].reset_index(drop=True)

        # Pre-compute residuals as a numpy array once at construction time.
        # Shape: (N, 15).  Avoids repeated pandas column lookups in __getitem__.
        emp = self._df[self._EMP_COLS].values.astype(np.float32)
        par = self._df[self._PAR_COLS].values.astype(np.float32)
        self._residuals = emp - par   # shape (N, 15)

        # Keep metadata for inspection / debugging
        self.split      = split
        self.n_windows  = len(self._df)
        self.n_bins     = 15

        print(f"ResidualDataset(split='{split}'): {self.n_windows} windows  "
              f"| residual shape per item: ({self.n_bins},)")
        print(f"  residual range: [{self._residuals.min():.4f}, "
              f"{self._residuals.max():.4f}]")
        print(f"  residual mean : {self._residuals.mean():.5f}  "
              f"std: {self._residuals.std():.5f}")

    def __len__(self) -> int:
        return self.n_windows

    def __getitem__(self, idx: int) -> torch.Tensor:
        # Convert the pre-computed numpy row to a float32 tensor
        # shape: (15,)
        return torch.from_numpy(self._residuals[idx])

    # ── Convenience methods ───────────────────────────────────────────────
    def all_residuals(self) -> np.ndarray:
        """Return all residuals as a numpy array (N, 15) — useful for stats."""
        return self._residuals.copy()

    def metadata(self, idx: int) -> pd.Series:
        """Return the full metadata row for a given index."""
        return self._df.iloc[idx]


# ── Instantiate all three splits ──────────────────────────────────────────
ds_train = ResidualDataset(windows_df, "train")
ds_val   = ResidualDataset(windows_df, "val")
ds_test  = ResidualDataset(windows_df, "test")

# ── Sanity checks ─────────────────────────────────────────────────────────
print("\n── Sanity checks ────────────────────────────────────────────────────")

# 1. Length matches split counts in windows_df
for ds, split_name in [(ds_train,"train"),(ds_val,"val"),(ds_test,"test")]:
    expected = (windows_df["split"] == split_name).sum()
    assert len(ds) == expected, \
        f"Length mismatch for {split_name}: {len(ds)} vs {expected}"
print("  ✓ Dataset lengths match windows_df split counts")

# 2. __getitem__ returns correct shape and dtype
r_sample = ds_train[0]
assert r_sample.shape  == (15,),          f"Bad shape: {r_sample.shape}"
assert r_sample.dtype  == torch.float32,  f"Bad dtype: {r_sample.dtype}"
print(f"  ✓ __getitem__ returns shape {r_sample.shape}  dtype {r_sample.dtype}")

# 3. Residual equals emp − par
row0      = ds_train._df.iloc[0]
emp_check = row0[[f"hist_emp_{k:02d}" for k in range(15)]].values.astype(np.float32)
par_check = row0[[f"hist_par_{k:02d}" for k in range(15)]].values.astype(np.float32)
expected_r = emp_check - par_check
assert np.allclose(r_sample.numpy(), expected_r, atol=1e-6), \
    "Residual mismatch: __getitem__ != emp - par"
print("  ✓ Residual = hist_emp − hist_par (verified on first item)")

# 4. No NaNs
for ds, name in [(ds_train,"train"),(ds_val,"val"),(ds_test,"test")]:
    n_nan = np.isnan(ds.all_residuals()).sum()
    assert n_nan == 0, f"NaNs found in {name}: {n_nan}"
print("  ✓ No NaNs in any split")

# 5. Train and val residuals are disjoint by cycle
train_cycles = set(zip(ds_train._df["cycle"], ds_train._df["hemisphere"]))
val_cycles   = set(zip(ds_val._df["cycle"],   ds_val._df["hemisphere"]))
test_cycles  = set(zip(ds_test._df["cycle"],  ds_test._df["hemisphere"]))
assert len(train_cycles & val_cycles)  == 0, "Train/val cycle leak!"
assert len(train_cycles & test_cycles) == 0, "Train/test cycle leak!"
assert len(val_cycles   & test_cycles) == 0, "Val/test cycle leak!"
print("  ✓ No cycle-level leakage between splits")

# ── Visualisation: residual distribution across splits ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: histogram of all residual values per split
for ds, name, color in [
        (ds_train, "Train", "tab:blue"),
        (ds_val,   "Val",   "tab:orange"),
        (ds_test,  "Test",  "tab:green")]:
    axes[0].hist(ds.all_residuals().flatten(), bins=60,
                 color=color, alpha=0.55, density=True,
                 label=f"{name} (N={len(ds)})")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_xlabel("Residual density value")
axes[0].set_ylabel("Probability density")
axes[0].set_title("Distribution of residual values by split\n"
                   "Splits should overlap — same underlying distribution")
axes[0].legend()

# Right: mean residual profile per split
for ds, name, color, ls in [
        (ds_train, "Train", "tab:blue",   "-"),
        (ds_val,   "Val",   "tab:orange", "--"),
        (ds_test,  "Test",  "tab:green",  ":")]:
    mean_r = ds.all_residuals().mean(axis=0)
    axes[1].plot(LAT_CENTERS, mean_r, color=color, linewidth=2,
                 linestyle=ls, label=f"{name}")
    axes[1].fill_between(
        LAT_CENTERS,
        mean_r - ds.all_residuals().std(axis=0),
        mean_r + ds.all_residuals().std(axis=0),
        color=color, alpha=0.12
    )
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Latitude (°)")
axes[1].set_ylabel("Mean residual density")
axes[1].set_title("Mean residual profile by split  (± 1 std shaded)\n"
                   "Profiles should look similar — stratification check")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n✓ Task 33 complete")
print(f"  ds_train : {len(ds_train)} windows")
print(f"  ds_val   : {len(ds_val)} windows")
print(f"  ds_test  : {len(ds_test)} windows")
print(f"  Each __getitem__ returns: torch.float32 tensor of shape (15,)")
print(f"  Noise injection happens in the Lightning training step — not here.")


---
## Task 34 — Test and visualize the dataset

A dataset class is one of the easier objects to silently misimplement
because most bugs do not raise — they just return wrongly-shaped or
wrongly-typed tensors that surface as cryptic errors three layers deeper
in the training loop. Spend a few cells here on visual and numerical
sanity checks before moving on.

**Tasks:**
- Instantiate three datasets: `train_dataset`, `val_dataset`, `test_dataset`,
  using the `split` column. Print `len(...)` for each. The counts should
  match the values printed by the setup cell.
- Pull one sample from `train_dataset` (e.g. `train_dataset[0]`) and verify:
  - Its type is `torch.Tensor`.
  - Its shape is `(15,)`.
  - Its dtype is `torch.float32`.
  - It contains no `NaN` or `Inf` values.
- Pick four random rows from the training set and overplot them as bar
  charts on a 2 × 2 grid using `BIN_CENTERS` for the x-axis. They should
  look like residuals: roughly zero-mean across bins (because empirical
  and parametric densities both integrate to ~1 on the same grid), small
  at the edge bins (0–3° and 42–45°, near-empty in the data), and
  structured in the middle bins where the Spörer zone lives.
- Compute and print the bin-wise mean and standard deviation across the
  full training set. The bin-wise mean should be small (a few percent of
  density at most); the bin-wise standard deviation should be larger,
  reflecting the cycle-to-cycle structure the diffusion model will need
  to learn.


In [ ]:
# Put your code here for Task 34.
# Task 34: Test and visualize the dataset
# Depends on: ds_train, ds_val, ds_test (from Task 33)
#             LAT_CENTERS, BIN_WIDTH, N_BINS (from setup)

import torch
import numpy as np
import matplotlib.pyplot as plt

# ── Step 1: lengths ───────────────────────────────────────────────────────
print("── Dataset lengths ──────────────────────────────────────────────────")
for ds, name in [(ds_train, "train"), (ds_val, "val"), (ds_test, "test")]:
    expected = (windows_df["split"] == name).sum()
    match    = "✓" if len(ds) == expected else "✗"
    print(f"  {match} {name:<6}: {len(ds):>4} windows  "
          f"(windows_df has {expected})")

# ── Step 2: single-sample checks ─────────────────────────────────────────
print("\n── Single-sample checks (train_dataset[0]) ──────────────────────────")
sample = ds_train[0]

checks = [
    ("type is torch.Tensor",   isinstance(sample, torch.Tensor)),
    ("shape is (15,)",         sample.shape == (15,)),
    ("dtype is torch.float32", sample.dtype == torch.float32),
    ("no NaN values",          not torch.isnan(sample).any().item()),
    ("no Inf values",          not torch.isinf(sample).any().item()),
]
for desc, passed in checks:
    print(f"  {'✓' if passed else '✗'} {desc}")

print(f"\n  sample values : {sample.numpy().round(4)}")
print(f"  sample min    : {sample.min().item():.5f}")
print(f"  sample max    : {sample.max().item():.5f}")
print(f"  sample sum×BW : {(sample.numpy() * BIN_WIDTH).sum():.5f}  "
      f"(residual should integrate to ≈ 0)")

# ── Step 3: visual inspection — four random samples ───────────────────────
rng_34  = np.random.default_rng(7)
n_train = len(ds_train)
idx4    = rng_34.integers(0, n_train, size=4)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes_flat = axes.flatten()

for ax, idx in zip(axes_flat, idx4):
    r    = ds_train[int(idx)].numpy()
    meta = ds_train.metadata(int(idx))

    colors = ["tab:green" if v >= 0 else "tab:red" for v in r]
    ax.bar(LAT_CENTERS, r, width=BIN_WIDTH * 0.78,
           color=colors, alpha=0.78, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.8)

    # Mark the universal mean latitude for context
    mu_u = meta["mu_universal"]
    ax.axvline(mu_u, color="navy", linewidth=1.2, linestyle="--",
               alpha=0.7, label=f"μ_universal = {mu_u:.1f}°")

    ax.set_title(
        f"Cycle {int(meta['cycle'])} {meta['hemisphere']}  |  "
        f"τ = {meta['tau_center']:.2f} yr\n"
        f"amplitude = {meta['amplitude']:.0f} MSH  |  "
        f"n_obs = {int(meta['n_obs'])}  |  "
        f"window {meta['window_start'].date()}",
        fontsize=8
    )
    ax.set_xlabel("Latitude (°)", fontsize=8)
    ax.set_ylabel("Residual density", fontsize=8)
    ax.set_xlim(0, 45)
    ax.legend(fontsize=7)

    # Annotate sum × BW (should be near 0)
    integral = float((r * BIN_WIDTH).sum())
    ax.text(0.97, 0.04, f"∫r·dμ = {integral:.4f}",
            transform=ax.transAxes, ha="right", fontsize=7,
            color="gray")

fig.suptitle(
    "Four random training residuals\n"
    "Green = emp > par (more sunspots than model predicted)  |  "
    "Red = emp < par  |  Dashed = μ_universal",
    fontsize=9, y=1.01
)
plt.tight_layout()
plt.show()

# ── Step 4: bin-wise statistics across the full training set ──────────────
R_train = ds_train.all_residuals()          # shape (N_train, 15)
R_val   = ds_val.all_residuals()
R_test  = ds_test.all_residuals()

mean_train = R_train.mean(axis=0)           # shape (15,)
std_train  = R_train.std(axis=0)
se_train   = std_train / np.sqrt(len(ds_train))

print("\n── Bin-wise statistics (training set) ───────────────────────────────")
print(f"  {'Bin':>4}  {'Center':>7}  {'Mean':>10}  {'Std':>10}  "
      f"{'Mean/Std':>10}  {'|Mean|>2SE':>10}")
print("  " + "-" * 62)
for k in range(N_BINS):
    ratio    = mean_train[k] / std_train[k] if std_train[k] > 0 else 0
    sig_flag = "YES" if abs(mean_train[k]) > 2 * se_train[k] else "   "
    print(f"  {k:>4}  {LAT_CENTERS[k]:>7.1f}°  "
          f"{mean_train[k]:>10.5f}  {std_train[k]:>10.5f}  "
          f"{ratio:>10.3f}  {sig_flag:>10}")

print(f"\n  Overall:")
print(f"    max |mean|       : {np.abs(mean_train).max():.5f}")
print(f"    mean std         : {std_train.mean():.5f}")
print(f"    std range        : [{std_train.min():.5f}, {std_train.max():.5f}]")
print(f"    max |mean| / mean std: "
      f"{np.abs(mean_train).max() / std_train.mean():.3f}  "
      f"(< 0.5 = residuals are mostly zero-mean)")

# ── Step 5: compare statistics across splits ──────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 4))

# --- Mean residual profile ---
ax = axes2[0]
for R, name, color, ls in [
        (R_train, "Train", "tab:blue",   "-"),
        (R_val,   "Val",   "tab:orange", "--"),
        (R_test,  "Test",  "tab:green",  ":")]:
    m = R.mean(axis=0)
    s = R.std(axis=0) / np.sqrt(len(R))
    ax.plot(LAT_CENTERS, m, color=color, linewidth=2,
            linestyle=ls, label=name)
    ax.fill_between(LAT_CENTERS, m - 2*s, m + 2*s,
                    color=color, alpha=0.12)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Latitude (°)"); ax.set_ylabel("Mean residual density")
ax.set_title("Mean residual profile ± 2SE\n"
             "(should be near zero — parametric model is unbiased)")
ax.legend(fontsize=8)

# --- Std profile ---
ax2 = axes2[1]
for R, name, color, ls in [
        (R_train, "Train", "tab:blue",   "-"),
        (R_val,   "Val",   "tab:orange", "--"),
        (R_test,  "Test",  "tab:green",  ":")]:
    ax2.plot(LAT_CENTERS, R.std(axis=0), color=color, linewidth=2,
             linestyle=ls, label=name)
ax2.set_xlabel("Latitude (°)"); ax2.set_ylabel("Std of residual density")
ax2.set_title("Bin-wise std across splits\n"
              "(higher std = more cycle-to-cycle variability here)")
ax2.legend(fontsize=8)

# --- Distribution of all residual values ---
ax3 = axes2[2]
for R, name, color in [
        (R_train, "Train", "tab:blue"),
        (R_val,   "Val",   "tab:orange"),
        (R_test,  "Test",  "tab:green")]:
    ax3.hist(R.flatten(), bins=50, color=color, alpha=0.5,
             density=True, label=f"{name} (N={len(R)})")
ax3.axvline(0, color="black", linewidth=1)
ax3.set_xlabel("Residual density value")
ax3.set_ylabel("Probability density")
ax3.set_title("Distribution of all residual values\n"
              "(splits should overlap — same underlying distribution)")
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── Step 6: correlation structure of training residuals ────────────────────
cov_train  = np.cov(R_train.T)     # shape (15, 15)
corr_train = np.corrcoef(R_train.T)

fig3, axes3 = plt.subplots(1, 2, figsize=(13, 5))

from matplotlib.colors import TwoSlopeNorm
norm_corr = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

im0 = axes3[0].imshow(corr_train, cmap="RdBu_r", norm=norm_corr, aspect="auto")
axes3[0].set_title("Correlation matrix of training residuals\n"
                    "Off-diagonal structure = bins co-vary across windows")
plt.colorbar(im0, ax=axes3[0], fraction=0.046, pad=0.04)

ticks = np.arange(0, N_BINS, 3)
tick_labels = [f"{LAT_CENTERS[i]:.0f}°" for i in ticks]
for ax in axes3:
    ax.set_xticks(ticks); ax.set_xticklabels(tick_labels, fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(tick_labels, fontsize=7)

# Plot the leading eigenvector (dominant mode of variation)
eigvals, eigvecs = np.linalg.eigh(cov_train)
order            = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

axes3[1].bar(LAT_CENTERS, eigvecs[:, 0], width=BIN_WIDTH * 0.75,
             color=["tab:green" if v >= 0 else "tab:red"
                    for v in eigvecs[:, 0]], alpha=0.8)
axes3[1].axhline(0, color="black", linewidth=0.8)
axes3[1].set_xlabel("Latitude (°)")
axes3[1].set_ylabel("Eigenvector component")
axes3[1].set_title(
    f"Leading eigenvector of training residual covariance\n"
    f"Explains {eigvals[0]/eigvals.sum()*100:.1f}% of total variance — "
    f"dominant mode of cycle-to-cycle variation"
)

plt.tight_layout()
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────
print("\n── Summary ──────────────────────────────────────────────────────────")
print(f"  train: {len(ds_train)} windows  "
      f"| mean abs residual: {np.abs(R_train).mean():.5f}")
print(f"  val  : {len(ds_val)} windows  "
      f"| mean abs residual: {np.abs(R_val).mean():.5f}")
print(f"  test : {len(ds_test)} windows  "
      f"| mean abs residual: {np.abs(R_test).mean():.5f}")
print(f"\n  Leading eigenvalue explains "
      f"{eigvals[0]/eigvals.sum()*100:.1f}% of training variance")
print(f"  → diffusion model needs to learn this dominant mode "
      f"plus the remaining {eigvals[1:].sum()/eigvals.sum()*100:.1f}%")
print(f"\n✓ Task 34 complete — dataset verified, ready for DataLoaders (Task 35)")


---
## Task 35 — Build the DataLoaders

A `Dataset` is an indexed collection; a `DataLoader` is the object that
samples batches from it during training. The DataLoader's job is to handle
batching, shuffling, parallel data loading, and the iteration protocol the
Lightning trainer expects.

There is one design choice worth flagging that is not diffusion-specific
but is easy to get wrong: `shuffle=True` for the *training* loader and
`shuffle=False` for validation and test. The reason for shuffling at
training time is that stochastic gradient descent assumes batches are
roughly i.i.d. samples from the data distribution; if batches are returned
in a fixed order (especially if the data is sorted by cycle, as our
parquet roughly is), the gradient is biased per epoch and the model can
fit cycle-by-cycle artifacts. For validation and test, the order does not
affect the loss value; we leave shuffling off so that successive
validation runs produce identical batch orderings, which keeps val/test
metrics reproducible across epochs.

**Tasks:**
- Build three DataLoaders using `torch.utils.data.DataLoader`:
  - `train_loader`: `batch_size=64`, `shuffle=True`, `num_workers=0`
    (Colab is happiest with 0 here).
  - `val_loader`: `batch_size=64`, `shuffle=False`, `num_workers=0`.
  - `test_loader`: `batch_size=64`, `shuffle=False`, `num_workers=0`.
- Iterate one batch from each loader (`next(iter(loader))`) and verify:
  - The batch is a tensor of shape `(64, 15)` (or smaller for the last
    batch if the dataset size is not divisible by 64).
  - The dtype is `torch.float32`.
- Print the number of batches per epoch for each loader (`len(loader)`).

**Note for later:** with 15-dim data and batch size 64, a full pass through
the training set is fast — well under a second on Colab's GPU. We will
lean into this in Task 41 by training for many epochs.


In [ ]:
# Put your code here for Task 35.
# Task 35: Build the DataLoaders
# Depends on: ds_train, ds_val, ds_test (from Task 33)

from torch.utils.data import DataLoader

# ── Build the three loaders ───────────────────────────────────────────────
BATCH_SIZE = 64

train_loader = DataLoader(
    ds_train,
    batch_size  = BATCH_SIZE,
    shuffle     = True,       # essential for SGD — randomizes batch order
    num_workers = 0,          # 0 = main process only (Colab-safe)
    pin_memory  = DEVICE == "cuda",  # speeds up CPU→GPU transfer if on GPU
    drop_last   = False,      # keep the final partial batch
)

val_loader = DataLoader(
    ds_val,
    batch_size  = BATCH_SIZE,
    shuffle     = False,      # reproducible ordering for metrics
    num_workers = 0,
    pin_memory  = DEVICE == "cuda",
    drop_last   = False,
)

test_loader = DataLoader(
    ds_test,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = 0,
    pin_memory  = DEVICE == "cuda",
    drop_last   = False,
)

# ── Batch counts ──────────────────────────────────────────────────────────
print("── DataLoader summary ───────────────────────────────────────────────")
for loader, name in [
        (train_loader, "train"),
        (val_loader,   "val"),
        (test_loader,  "test")]:
    n_windows = len(loader.dataset)
    n_batches = len(loader)
    last_bs   = n_windows % BATCH_SIZE or BATCH_SIZE
    print(f"  {name:<6}: {n_windows:>4} windows  →  {n_batches:>3} batches/epoch  "
          f"(last batch size: {last_bs})")

# ── Step-through one batch from each loader ────────────────────────────────
print("\n── Batch verification ───────────────────────────────────────────────")
for loader, name in [
        (train_loader, "train"),
        (val_loader,   "val"),
        (test_loader,  "test")]:

    batch = next(iter(loader))

    # Expected shape: (min(BATCH_SIZE, len(dataset)), 15)
    expected_bs = min(BATCH_SIZE, len(loader.dataset))
    shape_ok    = (batch.shape[0] <= BATCH_SIZE and batch.shape[1] == 15)
    dtype_ok    = (batch.dtype == torch.float32)
    nan_ok      = not torch.isnan(batch).any().item()
    inf_ok      = not torch.isinf(batch).any().item()

    status = "✓" if all([shape_ok, dtype_ok, nan_ok, inf_ok]) else "✗"
    print(f"  {status} {name:<6}: shape={tuple(batch.shape)}  "
          f"dtype={batch.dtype}  "
          f"NaN={not nan_ok}  Inf={not inf_ok}")
    print(f"         min={batch.min().item():.5f}  "
          f"max={batch.max().item():.5f}  "
          f"mean={batch.mean().item():.5f}")

# ── Verify shuffle is working: two consecutive train batches differ ────────
print("\n── Shuffle verification (train loader) ──────────────────────────────")
it        = iter(train_loader)
batch_a   = next(it)
batch_b   = next(it)
identical = torch.equal(batch_a, batch_b)
print(f"  Consecutive batches identical: {identical}  "
      f"(should be False — shuffle is working if False)")

# ── Verify val loader is deterministic across two full passes ─────────────
print("\n── Determinism verification (val loader) ────────────────────────────")
first_pass  = torch.cat([b for b in val_loader], dim=0)
second_pass = torch.cat([b for b in val_loader], dim=0)
det_ok      = torch.equal(first_pass, second_pass)
print(f"  Two full val passes identical: {det_ok}  "
      f"(should be True — shuffle=False guarantees this)")

# ── Simulate what the training step receives ──────────────────────────────
print("\n── Training step simulation ─────────────────────────────────────────")
print("  The Lightning training_step receives one batch r of shape (B, 15).")
print("  Inside the step it will:")
print("    1. sample t  ~ Uniform{0, …, T}  — one per row  → shape (B,)")
print("    2. draw   ε  ~ N(0, I)           — one per row  → shape (B, 15)")
print("    3. compute r_t = α[t]·r + σ[t]·ε               → shape (B, 15)")
print("    4. predict ε̂  = network(r_t, t)                → shape (B, 15)")
print("    5. loss = MSE(ε̂, ε)")
print()

# Show what a simulated training step would look like with a real batch
r_batch = next(iter(train_loader))           # shape (B, 15)
B       = r_batch.shape[0]

# Sample random timesteps
rng_35  = torch.Generator(); rng_35.manual_seed(0)
t_batch = torch.randint(0, T + 1, (B,), generator=rng_35)  # shape (B,)

# Draw noise
eps_batch = torch.randn(B, 15)                              # shape (B, 15)

# Corrupt: need α[t] and σ[t] per row — shape (B, 1) for broadcasting
alpha_t = alpha_pt[t_batch].unsqueeze(1)                   # shape (B, 1)
sigma_t = sigma_pt[t_batch].unsqueeze(1)                   # shape (B, 1)
r_t     = alpha_t * r_batch + sigma_t * eps_batch          # shape (B, 15)

print(f"  Simulated training batch:")
print(f"    r      : {tuple(r_batch.shape)}  "
      f"range [{r_batch.min():.4f}, {r_batch.max():.4f}]")
print(f"    t      : {tuple(t_batch.shape)}  "
      f"range [{t_batch.min().item()}, {t_batch.max().item()}]")
print(f"    ε      : {tuple(eps_batch.shape)}  "
      f"mean {eps_batch.mean():.4f}  std {eps_batch.std():.4f}")
print(f"    r_t    : {tuple(r_t.shape)}  "
      f"range [{r_t.min():.4f}, {r_t.max():.4f}]")
print(f"    target : predict ε from (r_t, t) → MSE loss")

# ── Visualisation: one training batch ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Left: distribution of t values in one batch
axes[0].hist(t_batch.numpy(), bins=20, color="steelblue", alpha=0.75,
             edgecolor="white")
axes[0].set_xlabel("Sampled timestep t")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Timestep distribution in one batch (B={B})\n"
                   "Uniform over [0, T] — every noise level seen equally")

# Middle: α[t] and σ[t] values for this batch
axes[1].scatter(t_batch.numpy(), alpha_pt[t_batch].numpy(),
                s=12, alpha=0.6, color="tab:green", label="α_t")
axes[1].scatter(t_batch.numpy(), sigma_pt[t_batch].numpy(),
                s=12, alpha=0.6, color="tab:red",   label="σ_t")
axes[1].set_xlabel("Timestep t")
axes[1].set_ylabel("Schedule coefficient")
axes[1].set_title("α_t and σ_t for each batch element\n"
                   "Each row gets its own corruption level")
axes[1].legend(fontsize=8)

# Right: one clean vs one corrupted residual from the batch
k_show = 0
axes[2].bar(LAT_CENTERS - 0.6, r_batch[k_show].numpy(),
            width=BIN_WIDTH * 0.42, color="steelblue", alpha=0.75,
            label=f"r₀  (clean)")
axes[2].bar(LAT_CENTERS + 0.6, r_t[k_show].detach().numpy(),
            width=BIN_WIDTH * 0.42, color="tab:orange", alpha=0.75,
            label=f"r_t  (t={t_batch[k_show].item()}, "
                  f"SNR={snr[t_batch[k_show].item()]:.2f})")
axes[2].axhline(0, color="black", linewidth=0.7)
axes[2].set_xlabel("Latitude (°)")
axes[2].set_ylabel("Residual density")
axes[2].set_title("First batch element: clean vs corrupted\n"
                   "(what the network receives vs what it started from)")
axes[2].legend(fontsize=7.5)
axes[2].set_xlim(0, 45)

plt.tight_layout()
plt.show()

print(f"\n✓ Task 35 complete")
print(f"  train_loader: {len(train_loader)} batches/epoch  "
      f"(shuffle=True)")
print(f"  val_loader  : {len(val_loader)} batches/epoch  "
      f"(shuffle=False, deterministic)")
print(f"  test_loader : {len(test_loader)} batches/epoch  "
      f"(shuffle=False, deterministic)")
print(f"\n  Ready for Task 36 — sinusoidal timestep embedding")


---
## Task 36 — Implement the sinusoidal timestep embedding

We arrive at the first diffusion-specific piece of architecture. The model
trained in this notebook is a single network that has to handle every
noise level from t = 0 (clean data) to t = T-1 (essentially pure noise) —
200 different denoising tasks, all sharing weights. For one network to
behave differently at different noise levels, it needs to *know* which t
it is currently denoising. The mechanism that gives it that information
is called a **timestep embedding**: a learned representation of the integer
t that is fed into the network alongside r_t.

The standard recipe — borrowed wholesale from how Transformers encode
positions — is the **sinusoidal embedding**. Given an integer t and an
embedding dimension d, produce a d-dimensional vector whose components are
sines and cosines of t at logarithmically spaced frequencies. Different t
values produce vectors that point in different directions in this
d-dimensional space; the geometry is smooth (nearby t produce nearby
embeddings) and the frequencies span a wide range so that both fast and
slow variation in t can be represented.

The function below produces the raw sinusoidal embedding. Task 37 will
wrap it in a small learnable MLP. The embedding itself is fixed (no
learnable parameters); only the wrapper learns.

**Tasks:**
- Implement `sinusoidal_embedding(t, dim)` with the following contract:
  - `t` is a `torch.Tensor` of shape `(B,)` containing integer timesteps.
  - `dim` is the desired embedding dimension (an even integer; we will use
    64 in Task 37).
  - The function returns a `torch.Tensor` of shape `(B, dim)`.
- Use `einops.rearrange` for the broadcast that combines `t` and the
  frequency vector into a `(B, dim/2)` argument tensor. Do not use
  `unsqueeze`, `reshape`, or `view`.
- The frequency formula is `freqs[i] = exp(-log(10000) * i / (dim/2))` for
  `i` in `0, 1, …, dim/2 - 1`. The argument tensor is then
  `args[b, i] = t[b] * freqs[i]`. The embedding stacks `sin(args)` and
  `cos(args)` along the feature dimension.

**Visualization tasks:**
- Pick `dim=64` and compute the embedding for `t = [0, T//4, T//2, 3*T//4, T-1]`.
  Plot the five 64-dimensional vectors as bar charts on a single figure
  (one row per t value). The patterns should differ visibly — that visible
  difference is what allows a downstream MLP to behave differently at
  different t.
- For three chosen embedding dimensions (e.g. dim 0, 16, 32), plot the
  embedding value as a function of t for `t` ranging over `[0, T)`. Each
  plot should look like a sinusoid; the wavelength should grow with the
  dimension index, because higher dimension indices correspond to lower
  frequencies in this formula.


In [ ]:
# Put your code here for Task 36.
# Task 36: Sinusoidal timestep embedding
# Depends on: T, schedule (from setup), einops

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange, repeat

def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """
    Sinusoidal timestep embedding (fixed, no learnable parameters).

    Parameters
    ----------
    t   : torch.Tensor, shape (B,)  — integer timesteps
    dim : int  — embedding dimension, must be even

    Returns
    -------
    emb : torch.Tensor, shape (B, dim)
          First dim/2 components are sin, last dim/2 are cos.
    """
    assert dim % 2 == 0, f"dim must be even, got {dim}"
    assert t.ndim == 1,  f"t must be shape (B,), got {t.shape}"

    half = dim // 2

    # Frequency vector: shape (half,)
    # freqs[i] = exp(-log(10000) * i / half)
    # i=0 → freq=1 (fastest),  i=half-1 → freq=10000^{-1} (slowest)
    i     = torch.arange(half, dtype=torch.float32, device=t.device)
    freqs = torch.exp(-np.log(10000.0) * i / half)   # shape (half,)

    # Outer product t × freqs → args of shape (B, half)
    # Use einops: t is (B,), freqs is (half,)
    # rearrange t to (B, 1) and freqs to (1, half), then multiply
    t_col    = rearrange(t.float(), "b -> b 1")       # (B, 1)
    freq_row = rearrange(freqs,     "h -> 1 h")       # (1, half)
    args     = t_col * freq_row                        # (B, half)

    # Stack sin and cos along the feature dimension
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
    return emb


# ── Quick shape + dtype check ─────────────────────────────────────────────
DIM_EMB  = 64
t_test   = torch.tensor([0, T//4, T//2, 3*T//4, T-1])
emb_test = sinusoidal_embedding(t_test, DIM_EMB)

print("── Shape and dtype checks ───────────────────────────────────────────")
print(f"  Input  t : {tuple(t_test.shape)}  values: {t_test.tolist()}")
print(f"  Output   : {tuple(emb_test.shape)}  dtype: {emb_test.dtype}")
assert emb_test.shape == (5, DIM_EMB), f"Bad shape: {emb_test.shape}"
assert emb_test.dtype == torch.float32
print("  ✓ shape (5, 64)  ✓ dtype float32")

# Check that different t values produce different embeddings
for i in range(len(t_test)):
    for j in range(i+1, len(t_test)):
        assert not torch.equal(emb_test[i], emb_test[j]), \
            f"t={t_test[i]} and t={t_test[j]} produced identical embeddings!"
print("  ✓ all five t values produce distinct embeddings")

# Check range: sin/cos are bounded in [-1, 1]
print(f"  Embedding range: [{emb_test.min().item():.4f}, "
      f"{emb_test.max().item():.4f}]  (should be within [-1, 1])")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: Five embeddings as bar charts (one row per t)
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(5, 1, figsize=(16, 10), sharex=True)

t_labels  = [f"t = {v.item()}  ({v.item()/T*100:.0f}% through schedule)"
             for v in t_test]
dim_range = np.arange(DIM_EMB)

for ax, emb_row, label, t_val in zip(
        axes1, emb_test.numpy(), t_labels, t_test.tolist()):
    ax.bar(dim_range[:DIM_EMB//2], emb_row[:DIM_EMB//2],
           color="tab:blue", alpha=0.7, width=0.8, label="sin components")
    ax.bar(dim_range[DIM_EMB//2:], emb_row[DIM_EMB//2:],
           color="tab:orange", alpha=0.7, width=0.8, label="cos components")
    ax.axhline(0,  color="black", linewidth=0.6)
    ax.axvline(DIM_EMB//2 - 0.5, color="gray",
               linewidth=1, linestyle="--", alpha=0.5)
    ax.set_ylim(-1.1, 1.1)
    ax.set_ylabel("Value", fontsize=7)
    ax.set_title(
        f"{label}  |  SNR = {snr[t_val]:.3f}",
        fontsize=8
    )
    if ax == axes1[0]:
        ax.legend(fontsize=7, loc="upper right")

axes1[-1].set_xlabel("Embedding dimension index")
axes1[-1].set_xticks(np.arange(0, DIM_EMB, 8))

fig1.suptitle(
    f"Sinusoidal timestep embeddings (dim={DIM_EMB})\n"
    "Blue: sin components (dims 0–31)   Orange: cos components (dims 32–63)\n"
    "Each row corresponds to a different t — the patterns must differ "
    "visibly for the network to distinguish noise levels",
    fontsize=9, y=1.01
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: Embedding value as a function of t for selected dimensions
# ══════════════════════════════════════════════════════════════════════════
t_all   = torch.arange(0, T, dtype=torch.long)    # shape (T,)
emb_all = sinusoidal_embedding(t_all, DIM_EMB)    # shape (T, 64)

# Pick three sin dimensions and three cos dimensions
sin_dims = [0, 8, 16]   # fast, medium, slow
cos_dims = [32, 40, 48]
chosen_dims = sin_dims + cos_dims
half        = DIM_EMB // 2

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 6), sharey=True)
axes2_flat  = axes2.flatten()

for ax, d in zip(axes2_flat, chosen_dims):
    component = "sin" if d < half else "cos"
    freq_idx  = d if d < half else d - half
    freq_val  = np.exp(-np.log(10000.0) * freq_idx / half)
    wavelength_in_t = 2 * np.pi / freq_val

    ax.plot(t_all.numpy(), emb_all[:, d].numpy(),
            color="tab:blue" if d < half else "tab:orange",
            linewidth=1.5, alpha=0.9)
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.set_xlabel("Timestep t", fontsize=8)
    ax.set_ylabel("Embedding value", fontsize=8)
    ax.set_title(
        f"Dim {d}  ({component}[{freq_idx}])\n"
        f"freq = {freq_val:.5f}   wavelength ≈ {wavelength_in_t:.1f} steps",
        fontsize=8
    )
    ax.set_xlim(0, T)

fig2.suptitle(
    "Embedding value vs timestep t for selected dimensions\n"
    "Low dim index → high frequency (fast oscillation)   "
    "High dim index → low frequency (slow drift)\n"
    "Wavelength grows with dim index — this spans all timescales "
    "from fine-grained (dim 0) to coarse (dim 48)",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: Embedding matrix heatmap + pairwise distance
# ══════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(1, 2, figsize=(15, 5))

# Left: heatmap of the full (T, dim) embedding matrix
im = axes3[0].imshow(
    emb_all.numpy().T,     # shape (dim, T) so t is on x-axis
    aspect="auto", cmap="RdBu_r",
    vmin=-1, vmax=1,
    extent=[0, T, DIM_EMB, 0]
)
axes3[0].axhline(half, color="white", linewidth=1.5, linestyle="--",
                 alpha=0.7)
axes3[0].set_xlabel("Timestep t")
axes3[0].set_ylabel("Embedding dimension")
axes3[0].set_title(
    f"Full embedding matrix  (shape T × dim = {T} × {DIM_EMB})\n"
    "White dashed line separates sin (top) from cos (bottom) components"
)
plt.colorbar(im, ax=axes3[0], fraction=0.046, pad=0.04)

# Right: pairwise L2 distance between selected t embeddings
t_sample    = torch.arange(0, T, T//20)   # 20 evenly spaced timesteps
emb_sample  = sinusoidal_embedding(t_sample, DIM_EMB)
dists       = torch.cdist(emb_sample, emb_sample).numpy()

im2 = axes3[1].imshow(dists, aspect="auto", cmap="viridis")
axes3[1].set_title(
    "Pairwise L2 distance between embeddings\n"
    "Diagonal = 0,  off-diagonal should grow with |t_i − t_j|\n"
    "(smooth geometry = nearby t → nearby embeddings)"
)
n_sample = len(t_sample)
tick_step = max(1, n_sample // 5)
axes3[1].set_xticks(np.arange(0, n_sample, tick_step))
axes3[1].set_xticklabels(t_sample[::tick_step].numpy(), fontsize=7)
axes3[1].set_yticks(np.arange(0, n_sample, tick_step))
axes3[1].set_yticklabels(t_sample[::tick_step].numpy(), fontsize=7)
axes3[1].set_xlabel("Timestep t")
axes3[1].set_ylabel("Timestep t")
plt.colorbar(im2, ax=axes3[1], fraction=0.046, pad=0.04, label="L2 distance")

plt.tight_layout()
plt.show()

# ── Numeric summary ────────────────────────────────────────────────────────
print("\n── Numeric summary ──────────────────────────────────────────────────")
print(f"  dim = {DIM_EMB}  →  {half} sin dims + {half} cos dims")
print(f"  Frequency range: [{np.exp(-np.log(10000)*0/half):.4f}, "
      f"{np.exp(-np.log(10000)*(half-1)/half):.6f}]")
print(f"  Wavelength range: [{2*np.pi/np.exp(-np.log(10000)*0/half):.1f}, "
      f"{2*np.pi/np.exp(-np.log(10000)*(half-1)/half):.0f}] timesteps")
print(f"  T = {T} timesteps — longest wavelength >> T means the "
      f"slowest dimensions don't even complete one cycle")

# Mean pairwise distance between adjacent vs distant timesteps
adj_dists  = [float(torch.dist(sinusoidal_embedding(torch.tensor([t_]),   DIM_EMB),
                                sinusoidal_embedding(torch.tensor([t_+1]), DIM_EMB)))
              for t_ in range(0, T-1, 10)]
far_dists  = [float(torch.dist(sinusoidal_embedding(torch.tensor([t_]),       DIM_EMB),
                                sinusoidal_embedding(torch.tensor([t_+T//4]), DIM_EMB)))
              for t_ in range(0, 3*T//4, 10)]
print(f"\n  Mean L2 dist between adjacent t    : {np.mean(adj_dists):.4f}")
print(f"  Mean L2 dist between t and t+T//4 : {np.mean(far_dists):.4f}")
print(f"  Ratio (far/adj)                    : "
      f"{np.mean(far_dists)/np.mean(adj_dists):.2f}×  "
      f"(larger = more discriminable)")

print(f"\n✓ Task 36 complete — sinusoidal_embedding(t, dim) ready")
print(f"  Signature : sinusoidal_embedding(t: Tensor[B], dim: int) → Tensor[B, dim]")
print(f"  No learnable parameters — fixed geometric encoding of t")
print(f"  Task 37 will wrap this in a small MLP to make it learnable")

---
## Task 37 — Wrap the sinusoidal embedding in a learnable module

The raw sinusoidal embedding is fixed: it has no learnable parameters and
its representation of t is determined entirely by the formula. To let the
main network shape the t-representation it actually wants, we wrap the
sinusoidal embedding in a small MLP that *learns* to project the fixed
sinusoidal representation into a useful form. This wrapper is the
`TimestepEmbedding` module.

The wrapping pattern (fixed positional encoding → learnable projection)
is standard across Transformer-style architectures. It separates the
"how do I represent integer position as a vector" question (solved
analytically by the sinusoidal formula) from the "what t-information does
my downstream network find useful" question (solved by gradient descent
through the MLP).

**Tasks:**
- Implement `TimestepEmbedding(nn.Module)` with constructor arguments
  `embed_dim` (the dimension of the raw sinusoidal embedding, 64 by
  default) and `hidden_dim` (the dimension of the learned projection,
  128 by default).
- The constructor should build a small MLP:
  `Linear(embed_dim, hidden_dim) → SiLU → Linear(hidden_dim, hidden_dim)`.
  We use SiLU (also called Swish: x · sigmoid(x)) because it is the
  conventional activation in diffusion models and gives smoother gradients
  than ReLU at deep noise levels. ReLU also works; SiLU is just the
  standard pick.
- The `forward(self, t)` method takes `t` of shape `(B,)`, calls
  `sinusoidal_embedding(t, self.embed_dim)`, and returns the result of the
  MLP, of shape `(B, hidden_dim)`.

**Sanity check:** instantiate `TimestepEmbedding(embed_dim=64, hidden_dim=128)`,
construct a tensor `t = torch.arange(0, T, T // 8)` (shape `(8,)`), and
verify that `module(t)` returns a tensor of shape `(8, 128)` with no NaNs.


In [ ]:
# Put your code here for Task 37.
# Task 37: TimestepEmbedding module
# Depends on: sinusoidal_embedding (Task 36), torch, einops

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange

class TimestepEmbedding(nn.Module):
    """
    Wraps the fixed sinusoidal embedding in a small learnable MLP.

    Architecture:
        sinusoidal_embedding(t, embed_dim)        → (B, embed_dim)   [fixed]
        Linear(embed_dim, hidden_dim) → SiLU      → (B, hidden_dim)  [learned]
        Linear(hidden_dim, hidden_dim)             → (B, hidden_dim)  [learned]

    Parameters
    ----------
    embed_dim  : int  — dimension of the raw sinusoidal embedding (default 64)
    hidden_dim : int  — output dimension of the learned projection (default 128)
    """

    def __init__(self, embed_dim: int = 64, hidden_dim: int = 128):
        super().__init__()
        assert embed_dim % 2 == 0, f"embed_dim must be even, got {embed_dim}"

        self.embed_dim  = embed_dim
        self.hidden_dim = hidden_dim

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.SiLU(),                          # Swish: x · σ(x)
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        t : torch.Tensor, shape (B,)  — integer timesteps

        Returns
        -------
        emb : torch.Tensor, shape (B, hidden_dim)
        """
        # Fixed sinusoidal encoding — no gradient flows through this
        sin_emb = sinusoidal_embedding(t, self.embed_dim)   # (B, embed_dim)

        # Learnable projection — gradient flows through this
        return self.mlp(sin_emb)                             # (B, hidden_dim)

    def extra_repr(self) -> str:
        return (f"embed_dim={self.embed_dim}, "
                f"hidden_dim={self.hidden_dim}, "
                f"params={sum(p.numel() for p in self.parameters()):,}")


# ── Instantiate and inspect ────────────────────────────────────────────────
EMBED_DIM  = 64
HIDDEN_DIM = 128

t_emb_module = TimestepEmbedding(embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM)
print("── Module summary ───────────────────────────────────────────────────")
print(t_emb_module)
print(f"\nParameter count: "
      f"{sum(p.numel() for p in t_emb_module.parameters()):,}")
for name, p in t_emb_module.named_parameters():
    print(f"  {name:<30} shape={tuple(p.shape)}  "
          f"numel={p.numel()}")

# ── Sanity checks ─────────────────────────────────────────────────────────
print("\n── Sanity checks ────────────────────────────────────────────────────")
t_sanity = torch.arange(0, T, T // 8)   # shape (8,)
print(f"  Input t : {tuple(t_sanity.shape)}  values: {t_sanity.tolist()}")

t_emb_module.eval()
with torch.no_grad():
    out_sanity = t_emb_module(t_sanity)

checks = [
    ("output shape is (8, 128)",   out_sanity.shape == (8, HIDDEN_DIM)),
    ("output dtype is float32",    out_sanity.dtype == torch.float32),
    ("no NaN in output",           not torch.isnan(out_sanity).any().item()),
    ("no Inf in output",           not torch.isinf(out_sanity).any().item()),
    ("different t → different emb",
     all(not torch.equal(out_sanity[i], out_sanity[j])
         for i in range(8) for j in range(i+1, 8))),
]
for desc, passed in checks:
    print(f"  {'✓' if passed else '✗'} {desc}")

print(f"\n  Output shape : {tuple(out_sanity.shape)}")
print(f"  Output range : [{out_sanity.min().item():.4f}, "
      f"{out_sanity.max().item():.4f}]")
print(f"  Output mean  : {out_sanity.mean().item():.4f}")
print(f"  Output std   : {out_sanity.std().item():.4f}")

# ── Verify gradient flows through the MLP but not the sinusoidal part ─────
print("\n── Gradient flow check ──────────────────────────────────────────────")
t_grad = torch.arange(0, T, T // 8)
out_grad = t_emb_module(t_grad)
loss_dummy = out_grad.sum()
loss_dummy.backward()

for name, p in t_emb_module.named_parameters():
    has_grad = p.grad is not None
    grad_norm = p.grad.norm().item() if has_grad else 0.0
    print(f"  {name:<30} grad={'✓' if has_grad else '✗'}  "
          f"norm={grad_norm:.4f}")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: raw sinusoidal vs learned projection — side by side
# ══════════════════════════════════════════════════════════════════════════
t_all = torch.arange(0, T, dtype=torch.long)

# Raw sinusoidal (shape T, embed_dim)
with torch.no_grad():
    sin_all = sinusoidal_embedding(t_all, EMBED_DIM)
    mlp_all = t_emb_module(t_all)              # shape (T, hidden_dim)

fig1, axes1 = plt.subplots(1, 2, figsize=(15, 5))

im0 = axes1[0].imshow(
    sin_all.numpy().T, aspect="auto", cmap="RdBu_r",
    vmin=-1, vmax=1, extent=[0, T, EMBED_DIM, 0]
)
axes1[0].set_title(
    f"Raw sinusoidal embedding  (fixed, no parameters)\n"
    f"Shape: T × embed_dim = {T} × {EMBED_DIM}",
    fontsize=9
)
axes1[0].set_xlabel("Timestep t")
axes1[0].set_ylabel("Embedding dimension")
plt.colorbar(im0, ax=axes1[0], fraction=0.046, pad=0.04)

vmax_mlp = mlp_all.abs().quantile(0.98).item()
im1 = axes1[1].imshow(
    mlp_all.detach().numpy().T, aspect="auto", cmap="RdBu_r",
    vmin=-vmax_mlp, vmax=vmax_mlp, extent=[0, T, HIDDEN_DIM, 0]
)
axes1[1].set_title(
    f"Learned MLP projection  (random init — will change after training)\n"
    f"Shape: T × hidden_dim = {T} × {HIDDEN_DIM}",
    fontsize=9
)
axes1[1].set_xlabel("Timestep t")
axes1[1].set_ylabel("Hidden dimension")
plt.colorbar(im1, ax=axes1[1], fraction=0.046, pad=0.04)

fig1.suptitle(
    "TimestepEmbedding: fixed sinusoidal input → learned MLP output\n"
    "After training the right panel will encode the t-information "
    "the denoising network finds most useful",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: pairwise distances — does learning preserve sinusoidal geometry?
# ══════════════════════════════════════════════════════════════════════════
t_sample   = torch.arange(0, T, T // 20)
with torch.no_grad():
    sin_samp = sinusoidal_embedding(t_sample, EMBED_DIM)
    mlp_samp = t_emb_module(t_sample)

dist_sin = torch.cdist(sin_samp, sin_samp).numpy()
dist_mlp = torch.cdist(mlp_samp, mlp_samp).numpy()

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
for ax, dmat, title in [
        (axes2[0], dist_sin, "Pairwise L2: raw sinusoidal\n(fixed)"),
        (axes2[1], dist_mlp, "Pairwise L2: MLP projection\n(random init)")]:
    im = ax.imshow(dmat, aspect="auto", cmap="viridis")
    ax.set_title(title, fontsize=9)
    n = len(t_sample)
    step = max(1, n // 5)
    ax.set_xticks(np.arange(0, n, step))
    ax.set_xticklabels(t_sample[::step].numpy(), fontsize=7)
    ax.set_yticks(np.arange(0, n, step))
    ax.set_yticklabels(t_sample[::step].numpy(), fontsize=7)
    ax.set_xlabel("Timestep t"); ax.set_ylabel("Timestep t")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="L2 distance")

fig2.suptitle(
    "Pairwise distance structure before training\n"
    "Left (fixed): smooth gradient from diagonal outward — good geometry.\n"
    "Right (random init): scrambled — training will reshape this to be useful.",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: SiLU vs ReLU activation — why SiLU?
# ══════════════════════════════════════════════════════════════════════════
x     = torch.linspace(-4, 4, 200)
silu  = torch.nn.functional.silu(x)
relu  = torch.nn.functional.relu(x)
gelu  = torch.nn.functional.gelu(x)

fig3, ax3 = plt.subplots(figsize=(8, 4))
ax3.plot(x.numpy(), silu.numpy(),  color="tab:blue",   linewidth=2,
         label="SiLU (used here)  x·σ(x)")
ax3.plot(x.numpy(), relu.numpy(),  color="tab:orange", linewidth=2,
         linestyle="--", label="ReLU  max(0, x)")
ax3.plot(x.numpy(), gelu.numpy(),  color="tab:green",  linewidth=2,
         linestyle=":", label="GELU")
ax3.axhline(0, color="black", linewidth=0.7)
ax3.axvline(0, color="black", linewidth=0.7)
ax3.set_xlabel("Input x"); ax3.set_ylabel("Activation output")
ax3.set_title(
    "SiLU vs ReLU vs GELU\n"
    "SiLU is smooth everywhere (no kink at 0) and slightly negative "
    "for x < 0\n→ smoother gradients, standard in diffusion architectures"
)
ax3.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"\n✓ Task 37 complete")
print(f"  TimestepEmbedding(embed_dim={EMBED_DIM}, hidden_dim={HIDDEN_DIM})")
print(f"  Parameters: {sum(p.numel() for p in t_emb_module.parameters()):,}")
print(f"  forward(t: Tensor[B]) → Tensor[B, {HIDDEN_DIM}]")
print(f"  Ready for Task 38 — DiffusionMLP main network")


---
## Task 38 — Build the DiffusionMLP with an ablation flag

The main network: an MLP that takes (r_t, t), concatenates the timestep
embedding with r_t at the input, pushes through a stack of fully connected
layers, and returns ε̂ — the network's prediction of the noise that was
added. The output has the same shape as r_t.

The diffusion-specific architectural concern here is *how* t enters the
network. We use **input concatenation**: the timestep embedding is
appended to r_t at the input layer, and the MLP learns to read both pieces
from the concatenated vector. Two alternatives exist that are common in
image diffusion models — FiLM (feature-wise modulation, where t scales and
shifts feature maps at every layer) and cross-attention (where t serves as
a key/value pair attended to from r_t) — but for 15-dimensional data and
a 3-layer MLP, both are overkill. Input concatenation is the right level
of complexity for the residual problem.

The constructor takes a `use_timestep_embedding=True/False` flag. When
True, the network builds a TimestepEmbedding and concatenates its output
with r_t. When False, the network has no TimestepEmbedding, the input
dimension is just `data_dim=15`, and the network is t-blind — it produces
the same output regardless of t. The False setting is the **architectural
ablation** we will use in Task 44 to measure whether the timestep
embedding actually earns its keep on this dataset.

**Tasks:**
- Implement `DiffusionMLP(nn.Module)` with constructor arguments:
  - `data_dim=15` — the residual dimension.
  - `hidden_dim=128` — the MLP hidden width.
  - `t_embed_dim=64` — the sinusoidal embedding dimension.
  - `t_hidden_dim=128` — the TimestepEmbedding output dimension.
  - `n_layers=3` — the number of hidden layers in the main MLP.
  - `use_timestep_embedding=True` — the ablation flag.
- When `use_timestep_embedding=True`:
  - Build a `TimestepEmbedding(t_embed_dim, t_hidden_dim)` and store it.
  - Set the main MLP's input dimension to `data_dim + t_hidden_dim`.
- When `use_timestep_embedding=False`:
  - Do not build a TimestepEmbedding.
  - Set the main MLP's input dimension to just `data_dim`.
- Build the main MLP as `n_layers` hidden layers of `hidden_dim` units
  each, with SiLU activations between them, and a final
  `Linear(hidden_dim, data_dim)` output layer (no activation on the
  output — ε can take any real value).
- The `forward(self, r_t, t)` method:
  - Takes `r_t` of shape `(B, data_dim)` and `t` of shape `(B,)`.
  - When the flag is True, computes `t_emb = self.t_embedding(t)` and
    concatenates with `r_t` along the feature dimension to form the input.
  - When the flag is False, the input is just `r_t`.
  - Returns the MLP output, of shape `(B, data_dim)`.

The forward signature `(r_t, t) → eps_hat` is the contract the
LightningModule in Task 40 will rely on. Both the True and False
configurations satisfy this contract; in the False case, the network
simply ignores `t` entirely.


In [ ]:
# Put your code here for Task 38.
# Task 38: DiffusionMLP
# Depends on: TimestepEmbedding, sinusoidal_embedding (Tasks 36–37)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange

class DiffusionMLP(nn.Module):
    """
    Denoising MLP for the residual diffusion model.

    Takes (r_t, t) and returns eps_hat — the predicted noise.
    When use_timestep_embedding=False the network ignores t entirely
    and is used as the ablation baseline in Task 44.

    Architecture (use_timestep_embedding=True):
        t  → TimestepEmbedding → t_emb  shape (B, t_hidden_dim)
        [r_t ‖ t_emb]                   shape (B, data_dim + t_hidden_dim)
        → [Linear → SiLU] × n_layers   shape (B, hidden_dim)
        → Linear                         shape (B, data_dim)

    Architecture (use_timestep_embedding=False):
        r_t                             shape (B, data_dim)
        → [Linear → SiLU] × n_layers   shape (B, hidden_dim)
        → Linear                         shape (B, data_dim)

    Parameters
    ----------
    data_dim              : int   residual dimension (15)
    hidden_dim            : int   MLP hidden width (128)
    t_embed_dim           : int   sinusoidal embedding dim (64)
    t_hidden_dim          : int   TimestepEmbedding output dim (128)
    n_layers              : int   number of hidden layers (3)
    use_timestep_embedding: bool  ablation flag
    """

    def __init__(
        self,
        data_dim               : int  = 15,
        hidden_dim             : int  = 128,
        t_embed_dim            : int  = 64,
        t_hidden_dim           : int  = 128,
        n_layers               : int  = 3,
        use_timestep_embedding : bool = True,
    ):
        super().__init__()

        self.data_dim               = data_dim
        self.hidden_dim             = hidden_dim
        self.use_timestep_embedding = use_timestep_embedding

        # ── Timestep embedding (optional) ─────────────────────────────────
        if use_timestep_embedding:
            self.t_embedding = TimestepEmbedding(
                embed_dim  = t_embed_dim,
                hidden_dim = t_hidden_dim,
            )
            mlp_input_dim = data_dim + t_hidden_dim
        else:
            self.t_embedding = None
            mlp_input_dim    = data_dim

        # ── Main MLP ──────────────────────────────────────────────────────
        # Input layer
        layers = [
            nn.Linear(mlp_input_dim, hidden_dim),
            nn.SiLU(),
        ]
        # Hidden layers
        for _ in range(n_layers - 1):
            layers += [
                nn.Linear(hidden_dim, hidden_dim),
                nn.SiLU(),
            ]
        # Output layer — no activation, ε̂ is unbounded
        layers.append(nn.Linear(hidden_dim, data_dim))

        self.mlp = nn.Sequential(*layers)

    def forward(self, r_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        r_t : torch.Tensor, shape (B, data_dim)   — noisy residual
        t   : torch.Tensor, shape (B,)             — integer timesteps

        Returns
        -------
        eps_hat : torch.Tensor, shape (B, data_dim) — predicted noise
        """
        if self.use_timestep_embedding:
            t_emb = self.t_embedding(t)                     # (B, t_hidden_dim)
            x     = torch.cat([r_t, t_emb], dim=-1)        # (B, data_dim + t_hidden_dim)
        else:
            x = r_t                                          # (B, data_dim)

        return self.mlp(x)                                   # (B, data_dim)

    def extra_repr(self) -> str:
        return (f"data_dim={self.data_dim}, "
                f"hidden_dim={self.hidden_dim}, "
                f"use_timestep_embedding={self.use_timestep_embedding}, "
                f"params={sum(p.numel() for p in self.parameters()):,}")


# ── Instantiate both variants ─────────────────────────────────────────────
model_with_t    = DiffusionMLP(use_timestep_embedding=True)
model_without_t = DiffusionMLP(use_timestep_embedding=False)

print("── Model WITH timestep embedding ────────────────────────────────────")
print(model_with_t)
n_with    = sum(p.numel() for p in model_with_t.parameters())
n_without = sum(p.numel() for p in model_without_t.parameters())
print(f"\nTotal parameters: {n_with:,}")

print("\n── Model WITHOUT timestep embedding (ablation) ──────────────────────")
print(model_without_t)
print(f"\nTotal parameters: {n_without:,}")
print(f"\nParameter overhead of timestep embedding: "
      f"{n_with - n_without:,}  "
      f"({(n_with - n_without)/n_with*100:.1f}% of model)")

# ── Sanity checks ─────────────────────────────────────────────────────────
print("\n── Sanity checks ────────────────────────────────────────────────────")
B        = 64
r_t_test = torch.randn(B, 15)
t_test   = torch.randint(0, T+1, (B,))

for model, name in [(model_with_t,    "with    t-emb"),
                    (model_without_t, "without t-emb")]:
    model.eval()
    with torch.no_grad():
        eps_hat = model(r_t_test, t_test)

    checks = [
        ("output shape (64, 15)",  eps_hat.shape == (B, 15)),
        ("output dtype float32",   eps_hat.dtype == torch.float32),
        ("no NaN",                 not torch.isnan(eps_hat).any().item()),
        ("no Inf",                 not torch.isinf(eps_hat).any().item()),
    ]
    status = "✓" if all(c[1] for c in checks) else "✗"
    results = "  ".join(f"{'✓' if c[1] else '✗'} {c[0]}" for c in checks)
    print(f"\n  [{name}]  {results}")
    print(f"    output range: [{eps_hat.min().item():.4f}, "
          f"{eps_hat.max().item():.4f}]  "
          f"std: {eps_hat.std().item():.4f}")

# ── Verify t-blind model gives SAME output regardless of t ────────────────
print("\n── Ablation flag verification ────────────────────────────────────────")
model_without_t.eval()
t_a = torch.zeros(B, dtype=torch.long)
t_b = torch.full((B,), T, dtype=torch.long)
with torch.no_grad():
    out_a = model_without_t(r_t_test, t_a)
    out_b = model_without_t(r_t_test, t_b)

t_blind_ok = torch.equal(out_a, out_b)
print(f"  t-blind model: same output for t=0 and t=T?  "
      f"{'✓ YES' if t_blind_ok else '✗ NO'}")

# Verify t-aware model gives DIFFERENT output for different t
model_with_t.eval()
with torch.no_grad():
    out_c = model_with_t(r_t_test, t_a)
    out_d = model_with_t(r_t_test, t_b)
t_aware_ok = not torch.equal(out_c, out_d)
print(f"  t-aware model: different output for t=0 vs t=T?  "
      f"{'✓ YES' if t_aware_ok else '✗ NO'}")

# ── Gradient flow check ───────────────────────────────────────────────────
print("\n── Gradient flow ────────────────────────────────────────────────────")
for model, name in [(model_with_t,    "with t-emb"),
                    (model_without_t, "without t-emb")]:
    model.train()
    model.zero_grad()
    eps_hat = model(r_t_test, t_test)
    loss    = eps_hat.pow(2).mean()
    loss.backward()

    all_have_grad = all(
        p.grad is not None and p.grad.norm().item() > 0
        for p in model.parameters()
    )
    print(f"  [{name}]: all params have nonzero grad?  "
          f"{'✓' if all_have_grad else '✗'}")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: Architecture diagram
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(1, 2, figsize=(14, 6))

for ax, use_t, name in [
        (axes1[0], True,  "WITH timestep embedding"),
        (axes1[1], False, "WITHOUT timestep embedding (ablation)")]:

    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(name, fontsize=9, pad=8)

    def box(ax, x, y, w, h, label, color="lightblue", fontsize=8):
        from matplotlib.patches import FancyBboxPatch
        ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h,
                                     boxstyle="round,pad=0.1",
                                     facecolor=color, edgecolor="gray",
                                     linewidth=1.2))
        ax.text(x, y, label, ha="center", va="center",
                fontsize=fontsize, wrap=True)

    def arrow(ax, x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="->", color="gray",
                                   lw=1.2))

    if use_t:
        # t branch
        box(ax, 2.5, 9.0, 2.0, 0.7, "t  (B,)", color="lightyellow")
        box(ax, 2.5, 7.5, 2.8, 0.7, "SinEmb → MLP\n(B, 128)",
            color="lightyellow")
        arrow(ax, 2.5, 8.65, 2.5, 7.85)

        # r_t branch
        box(ax, 7.0, 9.0, 2.0, 0.7, "r_t  (B,15)", color="lightgreen")

        # concat
        box(ax, 5.0, 6.2, 3.5, 0.7, "cat[r_t, t_emb]  (B, 143)",
            color="lightcoral", fontsize=7)
        arrow(ax, 2.5, 7.15, 3.8, 6.55)
        arrow(ax, 7.0, 8.65, 6.2,  6.55)

        # hidden layers
        for i, y in enumerate([4.9, 3.7, 2.5]):
            box(ax, 5.0, y, 3.2, 0.7,
                f"Linear({128}) → SiLU\n(B, 128)", fontsize=7)
            arrow(ax, 5.0, y + 0.55,
                  5.0, y + 0.55 - (1.2 if i < 2 else 0.85))

        # output
        box(ax, 5.0, 1.3, 3.0, 0.7, "Linear → (B, 15)\nε̂ output",
            color="lightblue")
        arrow(ax, 5.0, 1.85, 5.0, 2.15)

    else:
        # r_t only
        box(ax, 5.0, 9.0, 2.0, 0.7, "r_t  (B,15)", color="lightgreen")

        # hidden layers
        for i, y in enumerate([7.5, 6.3, 5.1]):
            box(ax, 5.0, y, 3.2, 0.7,
                f"Linear({128}) → SiLU\n(B, 128)", fontsize=7)
            arrow(ax, 5.0, y + 0.55,
                  5.0, y + 0.55 - (1.1 if i < 2 else 0.85))

        box(ax, 5.0, 3.9, 3.0, 0.7, "Linear → (B, 15)\nε̂ output",
            color="lightblue")
        arrow(ax, 5.0, 4.45, 5.0, 4.75)

        # t is ignored label
        box(ax, 2.0, 9.0, 2.0, 0.7, "t  (B,)", color="lightgray")
        ax.text(2.0, 8.3, "ignored ✗", ha="center", fontsize=8,
                color="gray", style="italic")

fig1.suptitle("DiffusionMLP architecture — with vs without timestep embedding",
              fontsize=10)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: Output distribution at random init — both models
# ══════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4))

for ax, model, name in [
        (axes2[0], model_with_t,    "WITH t-emb"),
        (axes2[1], model_without_t, "WITHOUT t-emb")]:

    model.eval()
    all_outputs = []
    with torch.no_grad():
        for t_val in [0, T//4, T//2, 3*T//4, T]:
            t_batch  = torch.full((100,), t_val, dtype=torch.long)
            r_batch  = torch.randn(100, 15)
            eps_pred = model(r_batch, t_batch)
            all_outputs.append(eps_pred.numpy())

    labels = ["t=0", "t=T/4", "t=T/2", "t=3T/4", "t=T"]
    colors = ["tab:blue","tab:orange","tab:green","tab:red","tab:purple"]
    for arr, label, color in zip(all_outputs, labels, colors):
        ax.hist(arr.flatten(), bins=40, alpha=0.45, density=True,
                color=color, label=label)

    ax.set_xlabel("ε̂ value (random init)")
    ax.set_ylabel("Density")
    ax.set_title(f"Output distribution at random init\n[{name}]")
    ax.legend(fontsize=7)

fig2.suptitle(
    "At random init both models output similar distributions regardless of t\n"
    "After training the t-aware model's output should depend on t — "
    "that's what Task 44 will test",
    fontsize=9
)
plt.tight_layout()
plt.show()

print(f"\n✓ Task 38 complete")
print(f"  model_with_t    : {n_with:,} parameters")
print(f"  model_without_t : {n_without:,} parameters")
print(f"  Both satisfy forward(r_t: Tensor[B,15], t: Tensor[B]) → Tensor[B,15]")
print(f"  Ready for Task 39 — full forward pass sanity check")


---
## Task 39 — Sanity-test the model, including the t-sensitivity check

Before training anything, two model bugs to rule out. The first is
mundane: shape mismatches at the input or output of the MLP, which will
raise a clear PyTorch error. The second is much more dangerous: a model
that *looks* fine — accepts (r_t, t), produces an output of the right
shape, trains without error — but silently ignores t even when
`use_timestep_embedding=True`. This can happen if the timestep embedding
is implemented but its output is dropped before the concatenation, if the
concatenation axis is wrong, or if the embedding produces all-zero or
all-NaN outputs. None of these failures raise. They only become visible
when the ablation comparison in Task 44 produces "the timestep embedding
doesn't help" — and at that point you cannot tell whether the embedding
is genuinely unhelpful or simply broken.

The defense is the **t-sensitivity check**: hold `r_t` fixed, vary `t`,
and verify that the model's output measurably changes. For
`use_timestep_embedding=True` the change should be non-trivial; for
`use_timestep_embedding=False` the change should be exactly zero (the
network has no path through which t can affect its output).

**Tasks:**
- Instantiate two models:
  - `model_full = DiffusionMLP(use_timestep_embedding=True)`.
  - `model_blind = DiffusionMLP(use_timestep_embedding=False)`.
- For each model:
  - Construct a random batch `r_t = torch.randn(8, 15)` and a random
    `t = torch.randint(0, T, (8,))`.
  - Run `model(r_t, t)` and confirm the output has shape `(8, 15)` with
    no NaNs.
  - Print the total parameter count
    (`sum(p.numel() for p in model.parameters())`). The full model should
    have *more* parameters than the blind model — the difference is exactly
    the parameter count of the TimestepEmbedding module, which the blind
    model does not build.
- **t-sensitivity check (the critical one):**
  - Construct a single fixed input `r_t_fixed = torch.randn(15)` (shape `(15,)`).
  - Construct five timesteps `t_values = torch.tensor([0, T//4, T//2, 3*T//4, T-1])`.
  - Use `einops.repeat(r_t_fixed, 'd -> n d', n=5)` to build a `(5, 15)`
    batch with identical r_t but different t. Do not use `expand` or `tile`.
  - For each model, compute the model output and then the pairwise L2
    distance between rows (a 5 × 5 matrix). For `model_full`, the
    off-diagonal entries should be visibly non-zero. For `model_blind`,
    they should be exactly zero
    (`torch.allclose(output_blind[i], output_blind[j])` should hold for
    any i, j).
  - Display the two 5 × 5 distance matrices as heatmaps side by side. The
    full-model heatmap should show off-diagonal structure; the
    blind-model heatmap should be uniformly zero off-diagonal (a flat
    color map).

If the full-model heatmap is also uniformly zero off-diagonal, your
TimestepEmbedding is broken. Fix it before proceeding to Task 40 — every
downstream task will silently misbehave otherwise.


In [ ]:
# Put your code here for Task 39.
# Task 39: Sanity-test the model + t-sensitivity check
# Depends on: DiffusionMLP, TimestepEmbedding, sinusoidal_embedding, T
# Uses einops.repeat — no expand, tile, or unsqueeze

import torch
import numpy as np
import matplotlib.pyplot as plt
from einops import repeat

# ── Instantiate both models ───────────────────────────────────────────────
model_full  = DiffusionMLP(use_timestep_embedding=True)
model_blind = DiffusionMLP(use_timestep_embedding=False)

model_full.eval()
model_blind.eval()

# ── Step 1: basic forward pass + shape checks ─────────────────────────────
print("── Forward pass checks ──────────────────────────────────────────────")
torch.manual_seed(0)
r_t_test = torch.randn(8, 15)
t_test   = torch.randint(0, T, (8,))

for model, name in [(model_full,  "model_full  (with t-emb)"),
                    (model_blind, "model_blind (no t-emb) ")]:
    with torch.no_grad():
        out = model(r_t_test, t_test)

    n_params = sum(p.numel() for p in model.parameters())
    checks   = [
        ("shape (8, 15)",   out.shape == (8, 15)),
        ("dtype float32",   out.dtype == torch.float32),
        ("no NaN",          not torch.isnan(out).any().item()),
        ("no Inf",          not torch.isinf(out).any().item()),
    ]
    all_ok = all(c[1] for c in checks)
    print(f"\n  {name}")
    print(f"    params : {n_params:,}")
    for desc, ok in checks:
        print(f"    {'✓' if ok else '✗'} {desc}")

n_full  = sum(p.numel() for p in model_full.parameters())
n_blind = sum(p.numel() for p in model_blind.parameters())
print(f"\n  Parameter overhead of t-embedding : {n_full - n_blind:,}  "
      f"({(n_full-n_blind)/n_full*100:.1f}% of full model)")

# ── Step 2: t-sensitivity check ───────────────────────────────────────────
print("\n── t-sensitivity check ──────────────────────────────────────────────")
torch.manual_seed(42)
r_t_fixed = torch.randn(15)                              # shape (15,)
t_values  = torch.tensor([0, T//4, T//2, 3*T//4, T-1])  # shape (5,)
t_labels  = [f"t={v.item()}" for v in t_values]

# Repeat r_t_fixed into a (5, 15) batch — same residual, five different t
# Using einops.repeat as required: no expand / tile
r_t_batch = repeat(r_t_fixed, "d -> n d", n=5)          # shape (5, 15)

print(f"  r_t_batch shape : {tuple(r_t_batch.shape)}  "
      f"(all 5 rows are identical)")
print(f"  t_values        : {t_values.tolist()}")
print(f"  Rows identical? : "
      f"{all(torch.equal(r_t_batch[0], r_t_batch[i]) for i in range(5))}")

# Forward pass for both models
with torch.no_grad():
    out_full  = model_full (r_t_batch, t_values)   # (5, 15)
    out_blind = model_blind(r_t_batch, t_values)   # (5, 15)

# Pairwise L2 distance matrices (5 × 5)
dist_full  = torch.cdist(out_full,  out_full ).numpy()   # (5, 5)
dist_blind = torch.cdist(out_blind, out_blind).numpy()   # (5, 5)

# ── Assertions ───────────────────────────────────────────────────────────
print("\n  model_full  (use_timestep_embedding=True)")
max_off_full = dist_full[~np.eye(5, dtype=bool)].max()
print(f"    max off-diagonal L2 dist : {max_off_full:.6f}  "
      f"{'✓ non-zero (t-sensitive)' if max_off_full > 1e-6 else '✗ ZERO — embedding broken!'}")

print("\n  model_blind (use_timestep_embedding=False)")
max_off_blind = dist_blind[~np.eye(5, dtype=bool)].max()
print(f"    max off-diagonal L2 dist : {max_off_blind:.6f}  "
      f"{'✓ exactly zero (t-blind)' if max_off_blind < 1e-6 else '✗ non-zero — t is leaking in!'}")

# Check every pair for model_blind
all_blind_equal = all(
    torch.allclose(out_blind[i], out_blind[j])
    for i in range(5) for j in range(5)
)
print(f"    torch.allclose for all pairs : "
      f"{'✓ True' if all_blind_equal else '✗ False'}")

# ── Detailed row comparison (full model) ─────────────────────────────────
print(f"\n  Full model output rows (each row = output for one t value):")
print(f"  {'t':>6}  {'out[0:4]':>40}  {'L2 from t=0':>14}")
for i, (t_val, row) in enumerate(zip(t_values.tolist(), out_full)):
    dist_from_0 = float(torch.dist(out_full[0], row))
    preview     = row[:4].numpy().round(4)
    print(f"  {t_val:>6}  {str(preview):>40}  {dist_from_0:>14.6f}")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: pairwise distance heatmaps — the key diagnostic
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(1, 2, figsize=(13, 5))

for ax, dist_mat, name, broken in [
        (axes1[0], dist_full,
         "model_full  (use_timestep_embedding=True)",
         max_off_full < 1e-6),
        (axes1[1], dist_blind,
         "model_blind (use_timestep_embedding=False)",
         False)]:  # blind should always be zero

    vmax = max(dist_mat.max(), 1e-6)
    im   = ax.imshow(dist_mat, cmap="viridis",
                     vmin=0, vmax=vmax, aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="L2 distance")

    ax.set_xticks(range(5)); ax.set_xticklabels(t_labels, rotation=30, fontsize=8)
    ax.set_yticks(range(5)); ax.set_yticklabels(t_labels, fontsize=8)
    ax.set_title(name, fontsize=8.5)

    # Annotate each cell with the distance value
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{dist_mat[i,j]:.3f}",
                    ha="center", va="center", fontsize=7,
                    color="white" if dist_mat[i,j] > vmax*0.5 else "black")

    if broken:
        ax.set_title(name + "\n⚠ EMBEDDING BROKEN — fix before Task 40",
                     fontsize=8, color="red")
    elif np.eye(5, dtype=bool).__invert__()[dist_mat > 1e-6].all() \
            and name.startswith("model_full"):
        pass  # expected off-diagonal non-zero
    elif not name.startswith("model_full"):
        ax.set_title(name + "\n✓ uniformly zero off-diagonal (correct)",
                     fontsize=8, color="green")

fig1.suptitle(
    "t-sensitivity check: same r_t, five different t values\n"
    "Left (with embedding): off-diagonal should be LARGE — "
    "different t → different output\n"
    "Right (without embedding): off-diagonal should be ZERO — "
    "t has no path into the network",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: output profiles — all five t values for model_full
# ══════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
LAT_CENTERS_39 = np.linspace(1.5, 43.5, 15)
BIN_WIDTH_39   = 3.0

colors_t = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

# Left: full model — outputs differ across t
ax = axes2[0]
for i, (t_val, color) in enumerate(zip(t_values.tolist(), colors_t)):
    ax.step(
        np.append(LAT_CENTERS_39 - 1.5, LAT_CENTERS_39[-1] + 1.5),
        np.append(out_full[i].numpy(), out_full[i].numpy()[-1]),
        color=color, linewidth=2, alpha=0.85,
        label=f"t = {t_val}  (SNR={snr[t_val]:.2f})",
        where="post"
    )
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xlabel("Latitude (°)"); ax.set_ylabel("ε̂ (predicted noise)")
ax.set_title("model_full  output per t  (same r_t input)\n"
             "Lines should diverge — different t → different ε̂")
ax.legend(fontsize=7.5)

# Right: blind model — outputs should be identical
ax2 = axes2[1]
for i, (t_val, color) in enumerate(zip(t_values.tolist(), colors_t)):
    ax2.step(
        np.append(LAT_CENTERS_39 - 1.5, LAT_CENTERS_39[-1] + 1.5),
        np.append(out_blind[i].numpy(), out_blind[i].numpy()[-1]),
        color=color, linewidth=2, alpha=0.85,
        linestyle="--",
        label=f"t = {t_val}",
        where="post"
    )
ax2.axhline(0, color="black", linewidth=0.7)
ax2.set_xlabel("Latitude (°)"); ax2.set_ylabel("ε̂ (predicted noise)")
ax2.set_title("model_blind output per t  (same r_t input)\n"
              "Lines should be IDENTICAL — t has no effect")
ax2.legend(fontsize=7.5)

plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: t-sensitivity as a function of t — full model
# ══════════════════════════════════════════════════════════════════════════
# Fix r_t, sweep t from 0 to T-1, measure L2 distance from the t=0 output
torch.manual_seed(7)
r_fixed_2 = torch.randn(15)
t_sweep   = torch.arange(0, T, dtype=torch.long)
r_sweep   = repeat(r_fixed_2, "d -> n d", n=len(t_sweep))

with torch.no_grad():
    out_sweep_full  = model_full (r_sweep, t_sweep)   # (T, 15)
    out_sweep_blind = model_blind(r_sweep, t_sweep)   # (T, 15)

dist_from_0_full  = torch.cdist(
    out_sweep_full,
    out_sweep_full[0:1].expand(len(t_sweep), -1)
).diagonal().numpy()

dist_from_0_blind = torch.cdist(
    out_sweep_blind,
    out_sweep_blind[0:1].expand(len(t_sweep), -1)
).diagonal().numpy()

fig3, ax3 = plt.subplots(figsize=(12, 4))
ax3.plot(t_sweep.numpy(), dist_from_0_full,
         color="tab:blue", linewidth=2, label="model_full (with t-emb)")
ax3.plot(t_sweep.numpy(), dist_from_0_blind,
         color="tab:red", linewidth=1.5, linestyle="--",
         label="model_blind (no t-emb)")
ax3.set_xlabel("Timestep t")
ax3.set_ylabel("L2 distance from output at t=0")
ax3.set_title(
    "t-sensitivity sweep: L2(output(t), output(t=0)) vs t\n"
    "Blue should be non-zero and vary with t.  "
    "Red should be flat at 0."
)
ax3.legend(fontsize=9)
ax3.set_xlim(0, T-1)
plt.tight_layout()
plt.show()

# ── Final verdict ─────────────────────────────────────────────────────────
print("\n── Verdict ──────────────────────────────────────────────────────────")
embedding_works = max_off_full  > 1e-4
blind_ok        = max_off_blind < 1e-6

print(f"  t-embedding is active and varying : "
      f"{'✓ PASS' if embedding_works else '✗ FAIL — fix before Task 40'}")
print(f"  t-blind model ignores t completely: "
      f"{'✓ PASS' if blind_ok else '✗ FAIL — t is leaking through'}")

if embedding_works and blind_ok:
    print(f"\n✓ Task 39 complete — both models ready for Task 40")
    print(f"  model_full  : {n_full:,} params  t-sensitive ✓")
    print(f"  model_blind : {n_blind:,} params  t-blind ✓")
else:
    print(f"\n✗ Fix the flagged issues before proceeding.")
    if not embedding_works:
        print("  Check: sinusoidal_embedding output, cat axis, "
              "TimestepEmbedding forward pass")
    if not blind_ok:
        print("  Check: DiffusionMLP forward — t should not "
              "enter the graph when use_timestep_embedding=False")


---
## Task 40 — Wrap the model in a LightningModule

The `DiffusionMLP` from Task 38 is a `nn.Module`: it knows how to compute
ε̂ from (r_t, t), but it does not know what to train on, what loss to
minimize, what optimizer to use, or how to use the noise schedule.
PyTorch Lightning's `LightningModule` is the wrapper that adds these
training-specific responsibilities. When you write a Lightning module,
you are answering five questions:

1. *What submodules does the model contain?* (`__init__`)
2. *What constants does it need that are not learnable?* (`register_buffer`)
3. *What does one training step look like?* (`training_step`)
4. *What does one validation step look like?* (`validation_step`)
5. *What optimizer trains it?* (`configure_optimizers`)

For diffusion, the diffusion-specific content sits almost entirely in
question 3 — the training step is where the schedule, the forward
corruption, and the MSE-on-ε loss all converge. The other four questions
have boilerplate-ish answers.

A specific design point: the schedule arrays α and σ from Task 28 of
Week 07 are *constants* (no learnable parameters) but they need to live
on the same device as the model and be saved with the checkpoint. The
PyTorch idiom for that is `register_buffer` — call it in `__init__` with
the precomputed schedule tensors. They will then sit alongside the
model's parameters in the state dict, follow the model to GPU
automatically, and not get updated by the optimizer.

The arrays `alpha_np` and `sigma_np` are already in scope from the setup
cell — same cosine schedule formula as Week 07, recomputed here so this
notebook is self-contained. We pass them as constructor arguments and
register them as buffers, rather than recomputing the cosine formula
inside `__init__`. This keeps the schedule choice visible at the call
site and makes the LightningModule's `training_step` schedule-agnostic:
swap `alpha_np`, `sigma_np` for arrays from a linear schedule and nothing
in the training step changes.

**Tasks:**
- Define a class `DiffusionLightning(pl.LightningModule)`.
- Constructor takes:
  - `model` — an instantiated `DiffusionMLP`.
  - `alpha`, `sigma` — the 1D arrays from the setup cell (each of length T+1).
  - `T` — the number of timesteps (200).
  - `lr=1e-3` — the Adam learning rate.
- In the constructor:
  - Store the model as `self.model`.
  - Convert `alpha` and `sigma` to `torch.float32` tensors and register
    them as buffers (named `'alpha'` and `'sigma'`).
  - Store `T` and `lr` as attributes.
- Implement `training_step(self, batch, batch_idx)`:
  - `batch` is a tensor of clean residuals, shape `(B, 15)`. Call it
    `r_clean`.
  - Sample `t = torch.randint(0, self.T, (B,), device=r_clean.device)`.
  - Draw `eps = torch.randn_like(r_clean)`.
  - Look up `alpha_t = self.alpha[t]` and `sigma_t = self.sigma[t]`,
    each of shape `(B,)`.
  - Use `einops.rearrange(alpha_t, 'b -> b 1')` (and similarly for sigma_t)
    so they broadcast cleanly against `r_clean` of shape `(B, 15)`.
  - Compute `r_t = alpha_t * r_clean + sigma_t * eps`.
  - Compute `eps_hat = self.model(r_t, t)`.
  - Compute `loss = F.mse_loss(eps_hat, eps)`.
  - Log via `self.log('train_loss', loss, prog_bar=True)`.
  - Return `loss`.
- Implement `validation_step(self, batch, batch_idx)`:
  - Same structure as `training_step`, but log as `'val_loss'` and do not
    set `prog_bar=True`.
- Implement `configure_optimizers(self)`:
  - Return `torch.optim.Adam(self.parameters(), lr=self.lr)`.

**Conceptual note worth pausing on:** the `training_step` is fully
*schedule-agnostic*. It looks up `self.alpha[t]` and `self.sigma[t]`
from the buffers and uses them; it does not know whether those values
came from a cosine, linear, or any other schedule formula. Swapping
schedules later (Week 09 onward, if you want to compare schedule
families) will only require changing what you pass to the constructor
— not a single line of `training_step` will change. This is the
separation of concerns we want to preserve as the pipeline grows.


In [ ]:
# Put your code here for Task 40.
# Task 40: DiffusionLightning — LightningModule wrapper
# Depends on: DiffusionMLP (Task 38), alpha_pt, sigma_pt, T (setup cell)
# Uses einops.rearrange — no unsqueeze or view

import torch
import torch.nn.functional as F
import pytorch_lightning as pl
from einops import rearrange

class DiffusionLightning(pl.LightningModule):
    """
    Lightning wrapper for the DiffusionMLP denoising model.

    Handles:
      - Schedule buffers (alpha, sigma) — constants, not parameters
      - Training step: sample t, corrupt r, predict eps, MSE loss
      - Validation step: same but logged separately, no grad
      - Optimizer: Adam with configurable lr

    Parameters
    ----------
    model : DiffusionMLP
        Instantiated denoising network (with or without t-embedding).
    alpha : np.ndarray, shape (T+1,)
        Signal coefficients from the cosine schedule.
    sigma : np.ndarray, shape (T+1,)
        Noise coefficients from the cosine schedule.
    T : int
        Number of diffusion timesteps (200).
    lr : float
        Adam learning rate (default 1e-3).
    """

    def __init__(
        self,
        model : "DiffusionMLP",
        alpha : "np.ndarray",
        sigma : "np.ndarray",
        T     : int   = 200,
        lr    : float = 1e-3,
    ):
        super().__init__()

        self.model = model
        self.T     = T
        self.lr    = lr

        # Register schedule arrays as non-learnable buffers.
        # They will:
        #   - follow the model to GPU automatically
        #   - be saved in checkpoints
        #   - NOT be updated by the optimizer
        self.register_buffer("alpha",
            torch.tensor(alpha, dtype=torch.float32))   # shape (T+1,)
        self.register_buffer("sigma",
            torch.tensor(sigma, dtype=torch.float32))   # shape (T+1,)

        # Save hyperparameters for Lightning's checkpoint/logging
        self.save_hyperparameters(ignore=["model"])

    # ── Shared corruption logic ───────────────────────────────────────────
    def _corrupt(self, r_clean: torch.Tensor):
        """
        Sample t, draw eps, corrupt r_clean to r_t.

        Returns (r_t, eps, t) — all on the same device as r_clean.
        """
        B      = r_clean.shape[0]
        device = r_clean.device

        # Sample one timestep per batch element
        t   = torch.randint(0, self.T, (B,), device=device)  # (B,)

        # Draw standard Gaussian noise — same shape as the residual
        eps = torch.randn_like(r_clean)                        # (B, 15)

        # Look up schedule coefficients and reshape for broadcasting
        # rearrange 'b -> b 1' turns (B,) into (B, 1) so it broadcasts
        # against r_clean of shape (B, 15)
        alpha_t = rearrange(self.alpha[t], "b -> b 1")        # (B, 1)
        sigma_t = rearrange(self.sigma[t], "b -> b 1")        # (B, 1)

        # Forward corruption: r_t = alpha_t * r + sigma_t * eps
        r_t = alpha_t * r_clean + sigma_t * eps               # (B, 15)

        return r_t, eps, t

    # ── Training step ─────────────────────────────────────────────────────
    def training_step(self, batch: torch.Tensor, batch_idx: int):
        """
        One gradient step.

        batch : torch.Tensor, shape (B, 15) — clean residuals from Dataset
        """
        r_clean = batch                                # (B, 15)
        r_t, eps, t = self._corrupt(r_clean)

        eps_hat = self.model(r_t, t)                  # (B, 15)
        loss    = F.mse_loss(eps_hat, eps)

        self.log("train_loss", loss,
                 prog_bar=True, on_step=False, on_epoch=True)
        return loss

    # ── Validation step ───────────────────────────────────────────────────
    def validation_step(self, batch: torch.Tensor, batch_idx: int):
        """
        One validation forward pass (no gradient update).

        batch : torch.Tensor, shape (B, 15) — clean residuals from Dataset
        """
        r_clean = batch
        r_t, eps, t = self._corrupt(r_clean)

        eps_hat = self.model(r_t, t)
        loss    = F.mse_loss(eps_hat, eps)

        self.log("val_loss", loss,
                 prog_bar=False, on_step=False, on_epoch=True)
        return loss

    # ── Optimizer ─────────────────────────────────────────────────────────
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


# ── Instantiate ───────────────────────────────────────────────────────────
lit_full  = DiffusionLightning(
    model = DiffusionMLP(use_timestep_embedding=True),
    alpha = alpha,    # numpy arrays from setup cell
    sigma = sigma,
    T     = T,
    lr    = 1e-3,
)
lit_blind = DiffusionLightning(
    model = DiffusionMLP(use_timestep_embedding=False),
    alpha = alpha,
    sigma = sigma,
    T     = T,
    lr    = 1e-3,
)

print("── LightningModule summary ──────────────────────────────────────────")
for lm, name in [(lit_full, "lit_full"), (lit_blind, "lit_blind")]:
    n_model   = sum(p.numel() for p in lm.model.parameters())
    n_total   = sum(p.numel() for p in lm.parameters())
    n_buffers = sum(b.numel() for b in [lm.alpha, lm.sigma])
    print(f"\n  {name}")
    print(f"    model params  : {n_model:,}")
    print(f"    total params  : {n_total:,}  (should equal model params)")
    print(f"    buffer items  : alpha({lm.alpha.shape}), sigma({lm.sigma.shape})")
    print(f"    buffer numel  : {n_buffers:,}  (NOT in optimizer)")
    print(f"    lr            : {lm.lr}")
    print(f"    T             : {lm.T}")

# ── Verify buffers are on correct device and dtype ───────────────────────
print("\n── Buffer checks ────────────────────────────────────────────────────")
for lm, name in [(lit_full, "lit_full"), (lit_blind, "lit_blind")]:
    checks = [
        ("alpha dtype float32",   lm.alpha.dtype == torch.float32),
        ("sigma dtype float32",   lm.sigma.dtype == torch.float32),
        ("alpha shape (T+1,)",    lm.alpha.shape == (T+1,)),
        ("sigma shape (T+1,)",    lm.sigma.shape == (T+1,)),
        ("alpha[0] ≈ 1.0",        abs(lm.alpha[0].item() - 1.0) < 1e-4),
        ("sigma[T] ≈ 1.0",        abs(lm.sigma[T].item() - 1.0) < 1e-3),
        ("alpha NOT in params",
         not any(lm.alpha is p for p in lm.parameters())),
    ]
    print(f"\n  {name}:")
    for desc, ok in checks:
        print(f"    {'✓' if ok else '✗'} {desc}")

# ── Simulate one training step manually ──────────────────────────────────
print("\n── Manual training step simulation ──────────────────────────────────")
torch.manual_seed(0)
dummy_batch = torch.randn(64, 15)   # simulates one DataLoader batch

lit_full.train()
loss_val = lit_full.training_step(dummy_batch, batch_idx=0)
print(f"  training_step output  : {loss_val.item():.6f}  "
      f"(scalar MSE loss)")
print(f"  loss dtype            : {loss_val.dtype}")
print(f"  loss requires_grad    : {loss_val.requires_grad}  "
      f"(True = gradient can flow)")

# Backward pass check
loss_val.backward()
n_with_grad = sum(
    1 for p in lit_full.parameters() if p.grad is not None)
n_total_p   = sum(1 for p in lit_full.parameters())
print(f"  params with grad after backward: "
      f"{n_with_grad}/{n_total_p}  "
      f"{'✓' if n_with_grad == n_total_p else '✗'}")

# Verify alpha and sigma have NO grad (they're buffers)
print(f"  alpha.requires_grad   : {lit_full.alpha.requires_grad}  "
      f"{'✓ False' if not lit_full.alpha.requires_grad else '✗ should be False'}")
print(f"  sigma.requires_grad   : {lit_full.sigma.requires_grad}  "
      f"{'✓ False' if not lit_full.sigma.requires_grad else '✗ should be False'}")

# ── Simulate optimizer step ────────────────────────────────────────────────
print("\n── Optimizer step simulation ─────────────────────────────────────────")
opt = lit_full.configure_optimizers()
print(f"  Optimizer type  : {type(opt).__name__}")
print(f"  Learning rate   : {opt.param_groups[0]['lr']}")
print(f"  Params in opt   : {sum(p.numel() for g in opt.param_groups for p in g['params']):,}")

# Take one step and verify loss decreases (statistically, not guaranteed)
lit_full.zero_grad()
torch.manual_seed(1)
dummy_batch2 = torch.randn(64, 15)
loss_before  = lit_full.training_step(dummy_batch2, 0)
loss_before.backward()
opt.step()
lit_full.zero_grad()

torch.manual_seed(1)
loss_after = lit_full.training_step(dummy_batch2, 0)
print(f"  Loss before one Adam step: {loss_before.item():.6f}")
print(f"  Loss after  one Adam step: {loss_after.item():.6f}")
print(f"  Loss changed: {not torch.isclose(loss_before, loss_after)}  "
      f"(optimizer is doing something)")

# ══════════════════════════════════════════════════════════════════════════
# Figure: schedule buffers stored in the module
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

t_range = np.arange(T+1)

ax = axes[0]
ax.plot(t_range, lit_full.alpha.numpy(), color="tab:blue",
        linewidth=2, label="α (signal)")
ax.plot(t_range, lit_full.sigma.numpy(), color="tab:red",
        linewidth=2, label="σ (noise)")
ax.set_xlabel("Timestep t")
ax.set_ylabel("Schedule coefficient")
ax.set_title("Schedule buffers stored in LightningModule\n"
             "(register_buffer — not learnable, follows model to GPU)")
ax.legend()

# Show the variance-preserving property from the stored buffers
vp = lit_full.alpha**2 + lit_full.sigma**2
ax2 = axes[1]
ax2.plot(t_range, vp.numpy(), color="tab:purple", linewidth=2)
ax2.axhline(1.0, color="black", linewidth=1, linestyle="--",
            alpha=0.6, label="Expected = 1.0")
ax2.set_xlabel("Timestep t")
ax2.set_ylabel("α² + σ²")
ax2.set_title("Variance-preserving check from stored buffers\n"
              f"max deviation from 1.0: {(vp-1).abs().max().item():.2e}")
ax2.legend()
ax2.set_ylim(0.9, 1.1)

plt.tight_layout()
plt.show()

print(f"\n✓ Task 40 complete")
print(f"  lit_full  : ready to train (use_timestep_embedding=True)")
print(f"  lit_blind : ready to train (use_timestep_embedding=False)")
print(f"  Both have alpha and sigma registered as non-learnable buffers")
print(f"  training_step is schedule-agnostic — swap schedule by changing")
print(f"  the alpha/sigma arrays passed to the constructor")
print(f"\n  Ready for Task 41 — training")


---
## Task 41 — Train the unconditional diffusion model

With the dataset, dataloaders, model, and Lightning module in place, the
training loop becomes a few lines of glue: instantiate everything, hand
to a `Trainer`, and call `fit`. This is the payoff of the AI/ML pipeline
abstraction — the wiring you spent Tasks 33–40 building lets the actual
training command be tiny.

We will use **Weights & Biases (WandB)** for experiment logging. WandB
gives you live loss curves, hyperparameter tracking, and a comparison
view across runs that will be especially useful when the ablation in
Task 44 produces a second run to compare against the first. If you have
not used WandB before, the first run will prompt you to log in (free
account, follow the URL it gives you).

**Tasks:**
- Instantiate `model_full = DiffusionMLP(use_timestep_embedding=True)`.
- Instantiate
  `lightning_full = DiffusionLightning(model_full, alpha=alpha_np, sigma=sigma_np, T=T)`.
- Create a `WandbLogger` with `project='butterflai-wk08'` and a clear run
  name like `'unconditional_full'`.
- Create a `pl.Trainer`:
  - `max_epochs=200` (15-dim data is small; epochs are fast).
  - `logger=wandb_logger`.
  - `accelerator='auto'`, `devices='auto'`.
  - `log_every_n_steps=10`.
- Call `trainer.fit(lightning_full, train_loader, val_loader)`.
- After training completes, save the model checkpoint to a known path
  (e.g. `'./ckpt_full.ckpt'`) using `trainer.save_checkpoint(...)`.
- Plot the training and validation loss curves from the logged history.
  Both should decrease, with the val loss eventually plateauing or
  rising slowly if the model overfits.

**Important callout — what loss does and does not tell you:** a decreasing
training loss is necessary but **not sufficient** for a good diffusion
model. The training loss is the MSE between the network's predicted ε
and the true ε; it measures how well the network estimates the noise
that was added during corruption. It does *not* directly measure whether
the samples drawn from the trained model look like the training data. A
model can have a low MSE-on-ε and still produce samples that are
visually wrong — for example, if the network has memorized the marginal
mean of ε but not the per-r_t structure. We will evaluate sample quality
directly in Task 43, after we have built a sampler in Task 42.


In [ ]:
# Put your code here for Task 41.
# Task 41: Train the unconditional diffusion model
# Depends on: DiffusionMLP, DiffusionLightning, train_loader, val_loader
#             alpha, sigma, T (from setup)

import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import (
    ModelCheckpoint, EarlyStopping, LearningRateMonitor)
import matplotlib.pyplot as plt
import numpy as np
import os

# ── Install wandb if needed ────────────────────────────────────────────────
try:
    import wandb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"],
                   check=True)
    import wandb

# ── 0. Hyperparameters ────────────────────────────────────────────────────
MAX_EPOCHS  = 200
LR          = 1e-3
BATCH_SIZE  = 64    # already set in train_loader
CKPT_FULL   = "./ckpt_full.ckpt"
PROJECT     = "butterflai-wk08"
RUN_NAME    = "unconditional_full"

# ── 1. Instantiate model + Lightning module ────────────────────────────────
torch.manual_seed(42)
model_full      = DiffusionMLP(use_timestep_embedding=True)
lightning_full  = DiffusionLightning(
    model = model_full,
    alpha = alpha,        # numpy arrays from setup cell
    sigma = sigma,
    T     = T,
    lr    = LR,
)

n_params = sum(p.numel() for p in lightning_full.parameters())
print(f"Model parameters : {n_params:,}")
print(f"Training windows : {len(ds_train)}")
print(f"Val windows      : {len(ds_val)}")
print(f"Epochs           : {MAX_EPOCHS}")
print(f"Batches/epoch    : {len(train_loader)}")

# ── 2. Logger ─────────────────────────────────────────────────────────────
wandb_logger = WandbLogger(
    project  = PROJECT,
    name     = RUN_NAME,
    log_model= False,     # don't upload model weights to wandb
)
# Log hyperparameters
wandb_logger.log_hyperparams({
    "max_epochs"             : MAX_EPOCHS,
    "lr"                     : LR,
    "batch_size"             : BATCH_SIZE,
    "use_timestep_embedding" : True,
    "n_params"               : n_params,
    "T"                      : T,
    "data_dim"               : 15,
    "hidden_dim"             : 128,
    "n_layers"               : 3,
})

# ── 3. Callbacks ──────────────────────────────────────────────────────────
checkpoint_cb = ModelCheckpoint(
    monitor   = "val_loss",
    mode      = "min",
    save_top_k= 1,
    filename  = "best-{epoch:03d}-{val_loss:.5f}",
    verbose   = False,
)
early_stop_cb = EarlyStopping(
    monitor  = "val_loss",
    patience = 40,        # stop if val_loss doesn't improve for 40 epochs
    mode     = "min",
    verbose  = True,
)
lr_monitor_cb = LearningRateMonitor(logging_interval="epoch")

# ── 4. Trainer ────────────────────────────────────────────────────────────
trainer = pl.Trainer(
    max_epochs         = MAX_EPOCHS,
    logger             = wandb_logger,
    callbacks          = [checkpoint_cb, early_stop_cb, lr_monitor_cb],
    accelerator        = "auto",
    devices            = "auto",
    log_every_n_steps  = 10,
    enable_progress_bar= True,
    deterministic      = False,   # True slows GPU ops; not needed here
)

# ── 5. Train ──────────────────────────────────────────────────────────────
print(f"\nStarting training — run '{RUN_NAME}' in project '{PROJECT}'")
print("WandB live dashboard will appear at the URL printed below.\n")

trainer.fit(lightning_full, train_loader, val_loader)

# ── 6. Save checkpoint ────────────────────────────────────────────────────
trainer.save_checkpoint(CKPT_FULL)
print(f"\nCheckpoint saved to : {CKPT_FULL}")
print(f"Best val_loss       : {checkpoint_cb.best_model_score:.6f}")
print(f"Best epoch          : {checkpoint_cb.best_model_path}")

# ── 7. Extract loss history from Lightning's logged metrics ───────────────
# Lightning stores metrics in trainer.callback_metrics (last epoch only)
# and in the logger. We reconstruct curves from the CSV log if available.

# Try to read from wandb run (most reliable)
try:
    run_id   = wandb_logger.experiment.id
    api      = wandb.Api()
    run      = api.run(f"{wandb_logger.experiment.entity}/"
                       f"{PROJECT}/{run_id}")
    history  = run.history(samples=MAX_EPOCHS)
    epochs_h     = history["_step"].values
    train_loss_h = history["train_loss"].dropna().values
    val_loss_h   = history["val_loss"].dropna().values
    print(f"\nRetrieved {len(train_loss_h)} train / "
          f"{len(val_loss_h)} val loss values from WandB")
    source = "wandb"
except Exception as e:
    print(f"WandB history retrieval failed ({e}) — "
          f"falling back to manual tracking")
    # Fallback: re-read from the Lightning CSV logger if wandb fails
    train_loss_h = []
    val_loss_h   = []
    source       = "manual"

# If wandb retrieval failed, re-train with a CSVLogger as fallback
if source == "manual" or len(train_loss_h) == 0:
    from pytorch_lightning.loggers import CSVLogger
    csv_logger = CSVLogger(".", name="diffusion_logs", version=0)

    torch.manual_seed(42)
    model_full2     = DiffusionMLP(use_timestep_embedding=True)
    lightning_full2 = DiffusionLightning(
        model=model_full2, alpha=alpha, sigma=sigma, T=T, lr=LR)

    trainer2 = pl.Trainer(
        max_epochs         = MAX_EPOCHS,
        logger             = csv_logger,
        callbacks          = [
            ModelCheckpoint(monitor="val_loss", mode="min",
                            save_top_k=1, verbose=False),
            EarlyStopping(monitor="val_loss", patience=40, mode="min"),
        ],
        accelerator        = "auto",
        devices            = "auto",
        log_every_n_steps  = 10,
        enable_progress_bar= True,
    )
    trainer2.fit(lightning_full2, train_loader, val_loader)
    trainer2.save_checkpoint(CKPT_FULL)

    import pandas as pd
    log_path = "./diffusion_logs/version_0/metrics.csv"
    if os.path.exists(log_path):
        metrics_df   = pd.read_csv(log_path)
        train_rows   = metrics_df[metrics_df["train_loss"].notna()]
        val_rows     = metrics_df[metrics_df["val_loss"].notna()]
        train_loss_h = train_rows["train_loss"].values
        val_loss_h   = val_rows["val_loss"].values
        print(f"Loaded {len(train_loss_h)} train / "
              f"{len(val_loss_h)} val epochs from CSV")
        lightning_full = lightning_full2   # use the re-trained model

# ── 8. Loss curve plots ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: linear scale ────────────────────────────────────────────────────
ax = axes[0]
if len(train_loss_h) > 0:
    ax.plot(train_loss_h, color="tab:blue",   linewidth=1.8,
            label="Train loss")
if len(val_loss_h) > 0:
    ax.plot(val_loss_h,   color="tab:orange", linewidth=1.8,
            linestyle="--", label="Val loss")

# Mark best epoch
if len(val_loss_h) > 0:
    best_ep  = int(np.argmin(val_loss_h))
    best_val = val_loss_h[best_ep]
    ax.axvline(best_ep, color="gray", linewidth=1, linestyle=":",
               alpha=0.7)
    ax.scatter([best_ep], [best_val], color="tab:red", s=60, zorder=5,
               label=f"Best val = {best_val:.5f} (ep {best_ep})")

ax.set_xlabel("Epoch"); ax.set_ylabel("MSE loss (nats)")
ax.set_title("Training and validation loss\n(linear scale)")
ax.legend(fontsize=8)

# ── Right: log scale (clearer for long training) ──────────────────────────
ax2 = axes[1]
if len(train_loss_h) > 0:
    ax2.semilogy(train_loss_h, color="tab:blue",   linewidth=1.8,
                 label="Train loss")
if len(val_loss_h) > 0:
    ax2.semilogy(val_loss_h,   color="tab:orange", linewidth=1.8,
                 linestyle="--", label="Val loss")

if len(val_loss_h) > 0:
    ax2.axvline(best_ep, color="gray", linewidth=1, linestyle=":",
                alpha=0.7)
    ax2.scatter([best_ep], [best_val], color="tab:red", s=60, zorder=5,
                label=f"Best val = {best_val:.5f}")

ax2.set_xlabel("Epoch"); ax2.set_ylabel("MSE loss (log scale)")
ax2.set_title("Training and validation loss\n(log scale)")
ax2.legend(fontsize=8)

fig.suptitle(
    f"DiffusionMLP training — {RUN_NAME}\n"
    f"{n_params:,} params  |  {len(ds_train)} train windows  |  "
    f"T={T}  lr={LR}",
    fontsize=9, y=1.01
)
plt.tight_layout()
plt.show()

# ── 9. Training summary ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  Training summary")
print("=" * 60)
final_train = float(train_loss_h[-1]) if len(train_loss_h) > 0 else float("nan")
final_val   = float(val_loss_h[-1])   if len(val_loss_h)   > 0 else float("nan")
print(f"  Epochs trained       : {len(train_loss_h)}")
print(f"  Final train loss     : {final_train:.6f}")
print(f"  Final val   loss     : {final_val:.6f}")
if len(val_loss_h) > 0:
    print(f"  Best val   loss      : {best_val:.6f}  (epoch {best_ep})")
    gap = final_val - final_train
    print(f"  Train/val gap        : {gap:+.6f}  "
          f"({'overfit' if gap > 0.01 else 'ok'})")
print(f"  Checkpoint saved to  : {CKPT_FULL}")
print("=" * 60)

# ── 10. What the loss does and does not tell you ──────────────────────────
print("\nInterpretation notes:")
print("  ✓ Decreasing train loss: network is learning to predict ε")
print("  ✓ Val loss tracking train: generalization is ok")
print("  ✗ Low MSE ≠ good samples: the loss measures ε-prediction accuracy")
print("    not sample quality. Task 42 builds the sampler; Task 43 checks")
print("    whether the distribution of generated residuals matches training.")
print(f"\n✓ Task 41 complete — ready for Task 42 (DDIM sampler)")


---
## Task 42 — Implement a DDIM-style sampler

Sampling is the *inference* counterpart of training. Training corrupts
clean residuals to noise: we take a clean r, pick a random t, and produce
r_t. Sampling reverses that path: we start from pure noise r_T ~ N(0, I)
and step backward through the schedule until we arrive at a clean r_0 —
a generated residual.

There is an asymmetry between training and sampling worth pausing on.
Training is *parallel* over t: every sample in a batch gets an independent
random t, all denoised in a single forward pass through the network.
Sampling is *sequential*: you must walk the schedule from t = T-1 down to
t = 0, calling the network at every step. With T = 200 timesteps, sampling
one batch of residuals takes 200 sequential forward passes; training one
batch takes one. This is why training scales pleasantly with batch size
but sampling does not.

The reverse step we implement is **deterministic DDIM** (Song, Meng,
Ermon 2020). DDIM is mathematically related to the more famous DDPM
(Ho, Jain, Abbeel 2020), but DDIM removes the per-step random noise and
makes the sampling process deterministic given the starting r_T. The
advantages: simpler code, easier debugging, identical sample quality with
many fewer steps if you want to skip timesteps. The DDIM step using the
α_t / σ_t parameterization from Week 07 is:

```
ε̂        = model(r_t, t)
r̂_0      = (r_t - σ_t · ε̂) / α_t              # estimate the clean residual
r_{t-1}  = α_{t-1} · r̂_0 + σ_{t-1} · ε̂        # step to t-1 along the same direction
```

Read the second line carefully: we use the *same* ε̂ from the model both
to estimate r̂_0 and to step to t-1. That is the essence of the
deterministic DDIM step.

**Tasks:**
- Implement a function
  `sample(lightning_module, batch_size, data_dim, device)` that returns
  a batch of generated residuals of shape `(batch_size, data_dim)`. The
  function should pull `alpha`, `sigma`, and `T` from the
  `lightning_module`'s buffers and attributes.
- The function should:
  - Set `lightning_module.eval()` and wrap the body in `torch.no_grad()`.
  - Initialize `r_t = torch.randn(batch_size, data_dim, device=device)`.
  - Loop `t` from `T - 1` down to `0`:
    - Construct
      `t_batch = torch.full((batch_size,), t, dtype=torch.long, device=device)`.
    - Compute `eps_hat = lightning_module.model(r_t, t_batch)`.
    - Look up `alpha_t = lightning_module.alpha[t]`,
      `sigma_t = lightning_module.sigma[t]` (scalars).
    - Compute `r_0_hat = (r_t - sigma_t * eps_hat) / alpha_t`.
    - If `t > 0`:
      `r_t = lightning_module.alpha[t-1] * r_0_hat + lightning_module.sigma[t-1] * eps_hat`.
    - If `t == 0`: `r_t = r_0_hat` (the final estimate is the clean sample).
  - Return `r_t`.

**Sanity check:** call `sample(lightning_full, batch_size=8, data_dim=15, device=device)`
and verify the output has shape `(8, 15)` with no NaNs. Plot the eight
samples as bar charts on a 2 × 4 grid using `BIN_CENTERS`. They should
look like residuals (zero-mean-ish, structure in middle bins, small at
edges) — but at this point all you can confirm is "they don't look
obviously broken." The quantitative comparison against the training
distribution is Task 43.

**Note on alternatives:** The stochastic DDPM step adds a fresh ε draw at
each timestep, producing different samples from the same starting r_T.
DDPM is the more conventional choice in the literature; DDIM is what we
use here for pedagogical simplicity. If you finish early and are curious,
add a `stochastic=True` flag and implement the DDPM step
`r_{t-1} = α_{t-1} · r̂_0 + sqrt(σ_{t-1}^2 - eta^2) · ε̂ + eta · z` where
`z ~ N(0, I)` and `eta` is the step's noise level (commonly η = σ_{t-1}
for full DDPM, η = 0 for DDIM).


In [ ]:
# Put your code here for Task 42.
# Task 42: DDIM sampler
# Depends on: lightning_full (Task 41), DiffusionMLP, T, DEVICE

import torch
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange

@torch.no_grad()
def sample(
    lightning_module,
    batch_size : int,
    data_dim   : int  = 15,
    device     : str  = "cpu",
    verbose    : bool = False,
) -> torch.Tensor:
    """
    DDIM deterministic sampler.

    Starts from r_T ~ N(0, I) and steps backward through the schedule
    using the trained network to estimate ε̂ at each step.

    Parameters
    ----------
    lightning_module : DiffusionLightning
        Trained Lightning module (contains .model, .alpha, .sigma, .T).
    batch_size : int
        Number of residuals to generate.
    data_dim   : int
        Residual dimension (15).
    device     : str
        'cpu' or 'cuda'.
    verbose    : bool
        If True, print progress every 50 steps.

    Returns
    -------
    r_t : torch.Tensor, shape (batch_size, data_dim)
        Generated residuals.
    """
    lightning_module.eval()
    lightning_module.to(device)

    # Pull schedule arrays from the module's buffers
    alpha = lightning_module.alpha   # shape (T+1,)
    sigma = lightning_module.sigma   # shape (T+1,)
    T     = lightning_module.T

    # ── Initialise from pure noise ────────────────────────────────────────
    r_t = torch.randn(batch_size, data_dim, device=device)  # r_T ~ N(0, I)

    # ── Reverse loop: t = T-1 down to 0 ──────────────────────────────────
    for t in range(T - 1, -1, -1):
        t_batch = torch.full(
            (batch_size,), t, dtype=torch.long, device=device)  # (B,)

        # Network prediction: ε̂ from (r_t, t)
        eps_hat = lightning_module.model(r_t, t_batch)           # (B, 15)

        # Scalar schedule values at this step
        alpha_t = alpha[t]   # scalar
        sigma_t = sigma[t]   # scalar

        # Estimate the clean residual from the noisy one
        r_0_hat = (r_t - sigma_t * eps_hat) / alpha_t           # (B, 15)

        if t > 0:
            # DDIM step: move to t-1 along the same ε̂ direction
            alpha_tm1 = alpha[t - 1]
            sigma_tm1 = sigma[t - 1]
            r_t = alpha_tm1 * r_0_hat + sigma_tm1 * eps_hat     # (B, 15)
        else:
            # Final step: the clean estimate is the generated sample
            r_t = r_0_hat                                        # (B, 15)

        if verbose and t % 50 == 0:
            print(f"  t={t:>3}  |r_t| mean={r_t.norm(dim=1).mean():.4f}  "
                  f"alpha={alpha_t.item():.4f}  sigma={sigma_t.item():.4f}")

    return r_t


# ── Sanity checks ─────────────────────────────────────────────────────────
print("── Sanity checks ────────────────────────────────────────────────────")
torch.manual_seed(0)

samples_8 = sample(lightning_full, batch_size=8, data_dim=15, device=DEVICE)

checks = [
    ("output shape (8, 15)", samples_8.shape == (8, 15)),
    ("output dtype float32", samples_8.dtype == torch.float32),
    ("no NaN",               not torch.isnan(samples_8).any().item()),
    ("no Inf",               not torch.isinf(samples_8).any().item()),
]
for desc, ok in checks:
    print(f"  {'✓' if ok else '✗'} {desc}")

print(f"\n  Sample stats:")
print(f"    min  : {samples_8.min().item():.5f}")
print(f"    max  : {samples_8.max().item():.5f}")
print(f"    mean : {samples_8.mean().item():.5f}")
print(f"    std  : {samples_8.std().item():.5f}")

# ── Verbose sampling for one sample to see trajectory ─────────────────────
print("\n── Sampling trajectory (one sample, verbose) ────────────────────────")
_ = sample(lightning_full, batch_size=1, data_dim=15,
           device=DEVICE, verbose=True)

# ── Determinism check: same seed → same samples ───────────────────────────
print("\n── Determinism check ────────────────────────────────────────────────")
torch.manual_seed(7)
s1 = sample(lightning_full, batch_size=4, data_dim=15, device=DEVICE)
torch.manual_seed(7)
s2 = sample(lightning_full, batch_size=4, data_dim=15, device=DEVICE)
det_ok = torch.allclose(s1, s2)
print(f"  Same seed → identical samples: {'✓' if det_ok else '✗'}")

# Different seeds → different samples
torch.manual_seed(0)
s3 = sample(lightning_full, batch_size=4, data_dim=15, device=DEVICE)
torch.manual_seed(99)
s4 = sample(lightning_full, batch_size=4, data_dim=15, device=DEVICE)
diff_ok = not torch.allclose(s3, s4)
print(f"  Different seeds → different samples: {'✓' if diff_ok else '✗'}")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: 2×4 grid of generated samples
# ══════════════════════════════════════════════════════════════════════════
LAT_CENTERS_42 = np.linspace(1.5, 43.5, 15)
BIN_WIDTH_42   = 3.0

torch.manual_seed(0)
samples_8_np = sample(lightning_full, batch_size=8,
                       data_dim=15, device=DEVICE).cpu().numpy()

fig1, axes1 = plt.subplots(2, 4, figsize=(16, 7), sharey=True)
axes1_flat  = axes1.flatten()

for i, (ax, s) in enumerate(zip(axes1_flat, samples_8_np)):
    colors = ["tab:green" if v >= 0 else "tab:red" for v in s]
    ax.bar(LAT_CENTERS_42, s, width=BIN_WIDTH_42 * 0.78,
           color=colors, alpha=0.78, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xlabel("Latitude (°)", fontsize=7)
    ax.set_title(f"Generated sample {i+1}\n"
                 f"∫r·dμ = {(s*BIN_WIDTH_42).sum():.4f}",
                 fontsize=8)
    ax.set_xlim(0, 45)
    if i % 4 == 0:
        ax.set_ylabel("Residual density", fontsize=7)

fig1.suptitle(
    "8 generated residuals from DDIM sampler\n"
    "Green = predicted excess sunspots vs parametric model  |  "
    "Red = deficit\n"
    "Task 43 will verify these match the training distribution quantitatively",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: Generated samples vs real training residuals
# ══════════════════════════════════════════════════════════════════════════
torch.manual_seed(42)
N_GEN      = len(ds_train)   # match training set size for fair comparison
samples_np = sample(lightning_full, batch_size=N_GEN,
                     data_dim=15, device=DEVICE).cpu().numpy()
R_train_np = ds_train.all_residuals()   # shape (N_train, 15)

fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5))

# Left: mean profile
ax = axes2[0]
ax.plot(LAT_CENTERS_42, R_train_np.mean(axis=0),
        color="tab:blue", linewidth=2, label="Training residuals")
ax.plot(LAT_CENTERS_42, samples_np.mean(axis=0),
        color="tab:orange", linewidth=2, linestyle="--",
        label="Generated samples")
ax.fill_between(LAT_CENTERS_42,
                R_train_np.mean(axis=0) - R_train_np.std(axis=0),
                R_train_np.mean(axis=0) + R_train_np.std(axis=0),
                color="tab:blue", alpha=0.12)
ax.fill_between(LAT_CENTERS_42,
                samples_np.mean(axis=0) - samples_np.std(axis=0),
                samples_np.mean(axis=0) + samples_np.std(axis=0),
                color="tab:orange", alpha=0.12)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xlabel("Latitude (°)")
ax.set_ylabel("Mean residual density")
ax.set_title("Mean profile ± 1 std\n(should overlap for a good model)")
ax.legend(fontsize=8)

# Middle: std profile
ax2 = axes2[1]
ax2.plot(LAT_CENTERS_42, R_train_np.std(axis=0),
         color="tab:blue", linewidth=2, label="Training")
ax2.plot(LAT_CENTERS_42, samples_np.std(axis=0),
         color="tab:orange", linewidth=2, linestyle="--", label="Generated")
ax2.set_xlabel("Latitude (°)")
ax2.set_ylabel("Std of residual density")
ax2.set_title("Bin-wise std\n(generated should match training)")
ax2.legend(fontsize=8)

# Right: distribution of all residual values
ax3 = axes2[2]
ax3.hist(R_train_np.flatten(), bins=50, color="tab:blue",
         alpha=0.55, density=True, label=f"Training (N={len(R_train_np)})")
ax3.hist(samples_np.flatten(), bins=50, color="tab:orange",
         alpha=0.55, density=True, label=f"Generated (N={len(samples_np)})")
ax3.axvline(0, color="black", linewidth=1)
ax3.set_xlabel("Residual density value")
ax3.set_ylabel("Probability density")
ax3.set_title("Distribution of all residual values\n"
              "(histograms should overlap)")
ax3.legend(fontsize=8)

fig2.suptitle(
    "Generated vs training residuals — visual comparison\n"
    f"(quantitative evaluation in Task 43)",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: Sampling trajectory — watch r_t evolve over steps
# ══════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def sample_with_trajectory(lightning_module, data_dim=15, device="cpu",
                            record_steps=None):
    """Like sample() but records r_t at selected timesteps."""
    if record_steps is None:
        record_steps = set(range(0, lightning_module.T, lightning_module.T//8))
        record_steps.add(0)

    lightning_module.eval()
    alpha = lightning_module.alpha
    sigma = lightning_module.sigma
    T     = lightning_module.T

    torch.manual_seed(5)
    r_t       = torch.randn(1, data_dim, device=device)
    snapshots = {}

    for t in range(T - 1, -1, -1):
        t_batch = torch.full((1,), t, dtype=torch.long, device=device)
        eps_hat = lightning_module.model(r_t, t_batch)
        alpha_t = alpha[t]; sigma_t = sigma[t]
        r_0_hat = (r_t - sigma_t * eps_hat) / alpha_t

        if t > 0:
            r_t = alpha[t-1] * r_0_hat + sigma[t-1] * eps_hat
        else:
            r_t = r_0_hat

        if t in record_steps:
            snapshots[t] = r_t.squeeze(0).cpu().numpy()

    return snapshots

snapshots = sample_with_trajectory(lightning_full, device=DEVICE)
snap_ts   = sorted(snapshots.keys(), reverse=True)

n_snaps = len(snap_ts)
fig3, axes3 = plt.subplots(2, (n_snaps+1)//2,
                            figsize=(16, 6), sharey=True)
axes3_flat  = axes3.flatten()

for ax, t_snap in zip(axes3_flat, snap_ts):
    r_snap = snapshots[t_snap]
    colors = ["tab:green" if v >= 0 else "tab:red" for v in r_snap]
    ax.bar(LAT_CENTERS_42, r_snap, width=BIN_WIDTH_42*0.75,
           color=colors, alpha=0.78)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(f"t = {t_snap}\nSNR = {snr[t_snap]:.3f}", fontsize=8)
    ax.set_xlabel("Latitude (°)", fontsize=7)
    ax.set_xlim(0, 45)

for ax in axes3_flat[n_snaps:]:
    ax.set_visible(False)

fig3.suptitle(
    "DDIM sampling trajectory — one generated residual at selected timesteps\n"
    "Starts as pure noise (high t) and converges to a structured residual (t=0)",
    fontsize=9
)
plt.tight_layout()
plt.show()

print(f"\n✓ Task 42 complete")
print(f"  sample() generates residuals via {T}-step DDIM reverse process")
print(f"  Deterministic: same torch seed → identical samples")
print(f"  Ready for Task 43 — quantitative distribution verification")


---
## Task 43 — Verify the sampled distribution matches the training distribution

The training loss tells you the network is learning *something*; it does
not tell you the network has learned the *right* thing. The direct
empirical question is: do samples from the trained model look like real
training residuals? At the level of distributional statistics, if the
model has learned the marginal residual distribution well, its samples
should match the training residuals on bin-wise mean, bin-wise standard
deviation, and bin-bin covariance structure.

This task asks you to compute and visualize that comparison. The
diagnostics are the same three you used in Task 31 of Week 07 to verify
the forward process — bin-wise mean, bin-wise standard deviation, and
covariance matrix — applied here to two distributions: the empirical
training residuals and the model-generated residuals.

**Tasks:**
- Sample 1000 residuals from the trained full model:
  `samples_full = sample(lightning_full, batch_size=1000, data_dim=15, device=device).cpu().numpy()`.
- Collect 1000 random training residuals into a numpy array of shape
  `(1000, 15)` (sample with replacement from `train_dataset` if needed).
- Compute the **bin-wise mean** for both, plot side-by-side as bar charts
  using `BIN_CENTERS`. They should agree within ~`1/sqrt(1000) ≈ 3%` of
  typical residual scale.
- Compute the **bin-wise standard deviation** for both, plot side-by-side.
  They should agree similarly.
- Compute the **bin-bin covariance matrix** (15 × 15) for both, plot
  side-by-side as heatmaps with the same color scale. The model's
  covariance should reproduce the diagonal (per-bin variance) and the
  dominant off-diagonal structure (correlations between neighboring bins,
  anti-correlations between distant bins) of the training data.
- **Visual overlay:** plot 20 random training residuals and 20 random
  model samples on the same axes (different colors). They should be
  visually indistinguishable — same overall envelope, same kind of
  structure in the middle bins, same near-zero behavior at edges. If the
  model samples are visibly smoother or more uniform than the training
  residuals, the model has under-learned. If they are visibly noisier or
  spikier, the model has under-trained or has a bug.
- Compute and report a single scalar summary: the **bin-wise MSE** between
  the sampled bin-wise mean and the training bin-wise mean. We will reuse
  this number in Task 44 to compare against the ablation.

**Caveat to flag:** because this week's model is *unconditional*, it is
matching the *marginal* training distribution — averaged across all cycles,
phases, and amplitudes. It is not learning to generate residuals
appropriate for any specific window. That is fine for this week; it is
exactly what the unconditional model is supposed to do. The check
"do generated residuals match the training-set marginals" is the right
test for the unconditional model.

**Optional ad-hoc inspection:** if you want to see what a sampled residual
"adds" to a classical prediction, pick any window from `windows_df`, look
up its `(amplitude, tau_center)`, evaluate `classical.density(A, tau, BIN_CENTERS)`
to get the classical density on the bin grid, integrate it to get the
binned classical histogram, and overlay it with `binned_classical + r_sample`
for a few sampled residuals. The combined predictions will *not*
systematically improve over the classical alone for this specific window —
the residual is a draw from the marginal, not targeted at this window's
amplitude or phase. Conditioning on (amplitude, mu_universal) in Week 09
is what makes the residual targeted; the corresponding `compute_global_nll`
value-add metric arrives there.


In [ ]:
# Put your code here for Task 43.
# Task 43: Verify sampled distribution matches training distribution
# Depends on: sample(), lightning_full, ds_train, T, DEVICE

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats

LAT_CENTERS_43 = np.linspace(1.5, 43.5, 15)
BIN_WIDTH_43   = 3.0
N_COMPARE      = 1000

# ── Step 1: generate 1000 samples ─────────────────────────────────────────
print(f"Generating {N_COMPARE} samples from trained model...")
torch.manual_seed(0)
samples_full = sample(
    lightning_full,
    batch_size = N_COMPARE,
    data_dim   = 15,
    device     = DEVICE,
).cpu().numpy()   # shape (1000, 15)
print(f"  Generated: {samples_full.shape}")

# ── Step 2: collect 1000 training residuals (sample with replacement) ─────
rng_43     = np.random.default_rng(42)
train_all  = ds_train.all_residuals()              # shape (N_train, 15)
idx_sample = rng_43.choice(len(train_all), size=N_COMPARE, replace=True)
train_sub  = train_all[idx_sample]                 # shape (1000, 15)
print(f"  Training residuals sampled: {train_sub.shape}")

# ── Step 3: compute statistics ────────────────────────────────────────────
mean_train  = train_sub.mean(axis=0)        # (15,)
mean_gen    = samples_full.mean(axis=0)

std_train   = train_sub.std(axis=0)
std_gen     = samples_full.std(axis=0)

cov_train   = np.cov(train_sub.T)           # (15, 15)
cov_gen     = np.cov(samples_full.T)

corr_train  = np.corrcoef(train_sub.T)
corr_gen    = np.corrcoef(samples_full.T)

# Standard error for reference lines (1/sqrt(N))
se_ref = 1.0 / np.sqrt(N_COMPARE)

print(f"\n── Statistics summary ───────────────────────────────────────────────")
print(f"  {'Stat':<18}  {'Training':>12}  {'Generated':>12}  {'|Δ|':>10}")
print("  " + "-" * 58)
for stat, tr, ge in [
        ("mean  max|μ|",   np.abs(mean_train).max(), np.abs(mean_gen).max()),
        ("std   mean",     std_train.mean(),          std_gen.mean()),
        ("std   max",      std_train.max(),            std_gen.max()),
        ("cov   diag mean",np.diag(cov_train).mean(), np.diag(cov_gen).mean()),
        ("cov   offdiag",  np.abs(cov_train - np.diag(np.diag(cov_train))).mean(),
                           np.abs(cov_gen   - np.diag(np.diag(cov_gen  ))).mean()),
]:
    print(f"  {stat:<18}  {tr:>12.5f}  {ge:>12.5f}  {abs(tr-ge):>10.5f}")

# ── Scalar summary: bin-wise MSE between means ────────────────────────────
mean_mse = float(np.mean((mean_gen - mean_train)**2))
print(f"\n  Bin-wise MSE(mean_gen, mean_train) : {mean_mse:.8f}")
print(f"  Sqrt of above (RMS error)          : {np.sqrt(mean_mse):.6f}")
print(f"  Reference: 2/sqrt(N) = {2*se_ref:.4f}  "
      f"(±2 SE of sampling noise)")
print(f"  Model bias: {'SMALL (good)' if np.sqrt(mean_mse) < 2*se_ref else 'LARGE (bias present)'}")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: Mean and Std comparison
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(2, 2, figsize=(14, 8))

width = BIN_WIDTH_43 * 0.42

# ── Top-left: mean bar chart side by side ─────────────────────────────────
ax = axes1[0, 0]
ax.bar(LAT_CENTERS_43 - width/2, mean_train, width=width,
       color="tab:blue",   alpha=0.75, label="Training")
ax.bar(LAT_CENTERS_43 + width/2, mean_gen,   width=width,
       color="tab:orange", alpha=0.75, label="Generated")
ax.axhline(0, color="black", linewidth=0.8)
# 2 SE reference band
ax.fill_between(LAT_CENTERS_43,
                mean_train - 2*std_train/np.sqrt(N_COMPARE),
                mean_train + 2*std_train/np.sqrt(N_COMPARE),
                color="tab:blue", alpha=0.12, label="±2 SE (train)")
ax.set_xlabel("Latitude (°)"); ax.set_ylabel("Mean residual density")
ax.set_title(f"Bin-wise mean\nMSE(means) = {mean_mse:.2e}")
ax.legend(fontsize=7.5)

# ── Top-right: mean difference ────────────────────────────────────────────
ax = axes1[0, 1]
diff_mean = mean_gen - mean_train
colors_dm = ["tab:green" if v >= 0 else "tab:red" for v in diff_mean]
ax.bar(LAT_CENTERS_43, diff_mean, width=BIN_WIDTH_43*0.75,
       color=colors_dm, alpha=0.78)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline( 2*std_train.mean()/np.sqrt(N_COMPARE), color="gray",
            linewidth=1, linestyle="--", alpha=0.6, label="±2 SE")
ax.axhline(-2*std_train.mean()/np.sqrt(N_COMPARE), color="gray",
            linewidth=1, linestyle="--", alpha=0.6)
ax.set_xlabel("Latitude (°)"); ax.set_ylabel("Generated − Training mean")
ax.set_title("Mean difference (gen − train)\n"
             "Bars within ±2SE = no significant bias")
ax.legend(fontsize=7.5)

# ── Bottom-left: std comparison ───────────────────────────────────────────
ax = axes1[1, 0]
ax.bar(LAT_CENTERS_43 - width/2, std_train, width=width,
       color="tab:blue",   alpha=0.75, label="Training")
ax.bar(LAT_CENTERS_43 + width/2, std_gen,   width=width,
       color="tab:orange", alpha=0.75, label="Generated")
ax.set_xlabel("Latitude (°)"); ax.set_ylabel("Std of residual density")
ax.set_title("Bin-wise standard deviation\n"
             "(generated should match training spread)")
ax.legend(fontsize=7.5)

# ── Bottom-right: std ratio ───────────────────────────────────────────────
ax = axes1[1, 1]
std_ratio = std_gen / np.maximum(std_train, 1e-8)
colors_sr = ["tab:green" if 0.8 < r < 1.2 else "tab:red"
             for r in std_ratio]
ax.bar(LAT_CENTERS_43, std_ratio, width=BIN_WIDTH_43*0.75,
       color=colors_sr, alpha=0.78)
ax.axhline(1.0, color="black", linewidth=1.2, linestyle="--",
           label="Perfect match (ratio=1)")
ax.axhspan(0.8, 1.2, color="gray", alpha=0.08, label="±20% band")
ax.set_xlabel("Latitude (°)"); ax.set_ylabel("std(generated) / std(training)")
ax.set_title("Std ratio  (gen / train)\n"
             "Green = within ±20%  |  <1 = under-dispersed  |  >1 = over-dispersed")
ax.legend(fontsize=7.5)
ax.set_ylim(0, 2.5)

fig1.suptitle(
    f"Mean and std comparison: generated vs training  (N={N_COMPARE})\n"
    f"Bin-wise MSE(means) = {mean_mse:.2e}  |  "
    f"RMS error = {np.sqrt(mean_mse):.5f}  |  "
    f"2/√N = {2*se_ref:.4f}",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: Covariance matrices
# ══════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 3, figsize=(17, 5))

cov_absmax = max(np.abs(cov_train).max(), np.abs(cov_gen).max())
norm_cov   = mcolors.TwoSlopeNorm(
    vmin=-cov_absmax, vcenter=0, vmax=cov_absmax)

ticks      = np.arange(0, 15, 3)
tick_labels= [f"{LAT_CENTERS_43[i]:.0f}°" for i in ticks]

for ax, mat, title in [
        (axes2[0], cov_train, "Covariance — Training"),
        (axes2[1], cov_gen,   "Covariance — Generated"),
        (axes2[2], cov_gen - cov_train, "Difference (gen − train)")]:

    if ax == axes2[2]:
        diff_max = np.abs(mat).max()
        norm_use = mcolors.TwoSlopeNorm(
            vmin=-diff_max, vcenter=0, vmax=diff_max)
    else:
        norm_use = norm_cov

    im = ax.imshow(mat, cmap="RdBu_r", norm=norm_use, aspect="auto")
    ax.set_title(title, fontsize=9)
    ax.set_xticks(ticks); ax.set_xticklabels(tick_labels, fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(tick_labels, fontsize=7)
    ax.set_xlabel("Bin"); ax.set_ylabel("Bin")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig2.suptitle(
    "Bin-bin covariance structure: generated vs training\n"
    "Left two panels should look similar.  "
    "Right panel (difference) should be near zero.",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: Visual overlay — 20 training vs 20 generated
# ══════════════════════════════════════════════════════════════════════════
fig3, ax3 = plt.subplots(figsize=(14, 5))

idx_20_train = rng_43.choice(len(train_all), size=20, replace=False)
idx_20_gen   = rng_43.choice(N_COMPARE,      size=20, replace=False)

for i, idx in enumerate(idx_20_train):
    r = train_all[idx]
    ax3.step(
        np.append(LAT_CENTERS_43 - 1.5, LAT_CENTERS_43[-1] + 1.5),
        np.append(r, r[-1]),
        color="tab:blue", linewidth=0.9, alpha=0.35,
        where="post",
        label="Training residuals" if i == 0 else "_"
    )

for i, idx in enumerate(idx_20_gen):
    r = samples_full[idx]
    ax3.step(
        np.append(LAT_CENTERS_43 - 1.5, LAT_CENTERS_43[-1] + 1.5),
        np.append(r, r[-1]),
        color="tab:orange", linewidth=0.9, alpha=0.35,
        where="post", linestyle="--",
        label="Generated samples" if i == 0 else "_"
    )

# Overlay means
ax3.plot(LAT_CENTERS_43, mean_train, color="tab:blue",
         linewidth=2.5, alpha=0.9, label="Training mean")
ax3.plot(LAT_CENTERS_43, mean_gen,   color="tab:orange",
         linewidth=2.5, alpha=0.9, linestyle="--", label="Generated mean")
ax3.axhline(0, color="black", linewidth=0.8)
ax3.set_xlabel("Latitude (°)")
ax3.set_ylabel("Residual density")
ax3.set_title(
    "Visual overlay: 20 training residuals (blue) vs 20 generated (orange)\n"
    "Solid thick = mean.  Faint step lines = individual samples.\n"
    "If model is good: same overall envelope, same edge behavior, "
    "same variability in middle bins"
)
ax3.legend(fontsize=8, loc="upper right")
ax3.set_xlim(0, 45)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 4: Marginal distributions per bin — KDE comparison
# ══════════════════════════════════════════════════════════════════════════
fig4, axes4 = plt.subplots(3, 5, figsize=(16, 8))
axes4_flat  = axes4.flatten()

ks_stats = []
for k in range(15):
    ax  = axes4_flat[k]
    tr  = train_sub[:, k]
    ge  = samples_full[:, k]

    # KDE
    x_range = np.linspace(
        min(tr.min(), ge.min()),
        max(tr.max(), ge.max()), 200)
    from scipy.stats import gaussian_kde
    kde_tr = gaussian_kde(tr, bw_method=0.3)
    kde_ge = gaussian_kde(ge, bw_method=0.3)

    ax.plot(x_range, kde_tr(x_range), color="tab:blue",
            linewidth=1.5, label="Train")
    ax.plot(x_range, kde_ge(x_range), color="tab:orange",
            linewidth=1.5, linestyle="--", label="Gen")
    ax.axvline(0, color="black", linewidth=0.5, alpha=0.4)

    # KS test
    ks_stat, ks_p = stats.ks_2samp(tr, ge)
    ks_stats.append(ks_stat)
    color = "green" if ks_p > 0.05 else "red"
    ax.set_title(
        f"Bin {k}  ({LAT_CENTERS_43[k]:.0f}°)\n"
        f"KS={ks_stat:.3f} p={ks_p:.3f}",
        fontsize=7, color=color
    )
    ax.set_xlabel(""); ax.set_ylabel("")
    if k == 0:
        ax.legend(fontsize=6)

fig4.suptitle(
    "Per-bin marginal distributions: training vs generated\n"
    "Green title = KS p>0.05 (not significantly different)  "
    "Red = distributions differ statistically",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ── Final summary ─────────────────────────────────────────────────────────
n_ks_pass = sum(1 for k in range(15)
                if stats.ks_2samp(train_sub[:,k],
                                   samples_full[:,k]).pvalue > 0.05)

print("\n" + "=" * 65)
print("  Task 43 Summary")
print("=" * 65)
print(f"  N generated  : {N_COMPARE}")
print(f"  N training   : {len(train_sub)}")
print(f"\n  Mean")
print(f"    MSE(mean_gen, mean_train) : {mean_mse:.8f}")
print(f"    RMS error                 : {np.sqrt(mean_mse):.6f}")
print(f"    2/√N reference            : {2*se_ref:.6f}")
print(f"\n  Std")
print(f"    Training mean std  : {std_train.mean():.5f}")
print(f"    Generated mean std : {std_gen.mean():.5f}")
print(f"    Ratio (gen/train)  : {std_gen.mean()/std_train.mean():.4f}  "
      f"(1.0 = perfect)")
print(f"\n  Covariance")
cov_offdiag_diff = np.abs((cov_gen - cov_train) -
                           np.diag(np.diag(cov_gen - cov_train))).mean()
print(f"    Mean abs off-diag diff : {cov_offdiag_diff:.6f}")
print(f"\n  Per-bin KS test")
print(f"    Bins passing (p>0.05)  : {n_ks_pass}/15")
print(f"    Mean KS statistic      : {np.mean(ks_stats):.4f}  "
      f"(0=identical, 1=completely different)")
print(f"\n  Scalar summary for Task 44 ablation:")
print(f"    mean_mse_full = {mean_mse:.8f}")
print("=" * 65)

# Store for Task 44
mean_mse_full = mean_mse
print(f"\n✓ Task 43 complete — mean_mse_full = {mean_mse_full:.8f}")
print(f"  Ready for Task 44 — ablation experiment")


---
## Task 44 — Run the no-timestep-embedding ablation

The whole reason for the ablation flag in Task 38 is the question this
task answers: does the timestep embedding earn its keep? We trained one
model with `use_timestep_embedding=True`. Now we train a second, identical
in every other respect, with `use_timestep_embedding=False`. The
**ablation comparison** is the two models' performance on the same metrics
from Task 43. If the t-blind model does almost as well, the timestep
embedding is not contributing much (or is broken — Task 39 should have
ruled the latter out). If the t-blind model does substantially worse,
the timestep embedding earns its keep.

There is a useful prediction to commit to before running this task. The
t-blind model has only one possible strategy: it must produce a single
ε estimate that is averaged over all noise levels — a kind of "average
denoiser" that does not adapt to the noise level it is currently looking
at. At low t, where the input is mostly clean, this average denoiser will
add too much noise to the estimate; at high t, where the input is mostly
noise, it will subtract too little. The net effect on samples is
typically that the marginal mean and variance roughly match (because the
overall scale is correct on average) but the covariance structure
collapses — the model loses the per-bin correlations that distinguish
training residuals from white noise.

Predict before you run. Then run.

**Tasks:**
- Instantiate `model_blind = DiffusionMLP(use_timestep_embedding=False)`.
- Instantiate
  `lightning_blind = DiffusionLightning(model_blind, alpha=alpha_np, sigma=sigma_np, T=T)`.
- Train it with the same Trainer configuration as Task 41, but with a
  distinct WandB run name like `'unconditional_blind'`. Use the same
  `max_epochs`, the same DataLoaders, the same learning rate. Save the
  checkpoint to `'./ckpt_blind.ckpt'`.
- Sample 1000 residuals from the trained blind model.
- Re-run all three diagnostics from Task 43 (bin-wise mean, std,
  covariance heatmap) for the blind model.
- Build a comparison table with three rows (training data, full model,
  blind model) and four columns (final train_loss, final val_loss,
  bin-wise mean MSE vs. training, qualitative covariance match yes/no).
- Plot the two covariance heatmaps (full and blind) side by side. The
  qualitative comparison is the headline result: does the blind model's
  covariance match the training data's, or has it collapsed to something
  more diagonal?

**How to read the result:**

- *If the full model clearly beats the blind model* on covariance match:
  the timestep embedding earns its keep. The architectural complexity is
  doing real work, and Week 09's conditioning machinery should be built
  on top of the t-aware base.
- *If the two models are nearly tied:* the timestep embedding is not
  contributing much on this dataset. Possible reasons: 200 timesteps is
  more than this 15-dimensional residual problem needs (a network can
  approximately memorize an "average denoiser" when the noise levels are
  not too varied); the cosine schedule keeps the signal alive across a
  large fraction of t (so the average denoiser is not far from any
  specific denoiser); or the data does not have enough structure across
  noise levels for t-awareness to matter. None of these is a bug — they
  are real findings about this problem at this scale.
- *If the blind model outperforms the full model:* a real bug somewhere,
  most likely in the TimestepEmbedding or in the concatenation. Re-run
  Task 39's t-sensitivity check to localize.

Either of the first two outcomes is publishable insight about your
specific problem. The result is not predetermined — that is what makes
this an ablation rather than a demonstration.


In [ ]:
# Put your code here for Task 44.
# Task 44: No-timestep-embedding ablation
# Depends on: DiffusionMLP, DiffusionLightning, train_loader, val_loader,
#             sample(), alpha, sigma, T, DEVICE, mean_mse_full (Task 43)

import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
import pandas as pd
import os

CKPT_BLIND   = "./ckpt_blind.ckpt"
RUN_NAME_BLIND = "unconditional_blind"
MAX_EPOCHS_44  = MAX_EPOCHS   # same as Task 41
LR_44          = LR

# ── Pre-run prediction (fill in before running) ───────────────────────────
print("── Pre-run prediction ───────────────────────────────────────────────")
print("  The t-blind model must produce a single ε estimate averaged over")
print("  all noise levels. Predicted outcome:")
print("    - Final loss     : similar to full model (average denoiser is")
print("                       not catastrophically wrong)")
print("    - Mean profile   : roughly matched (marginal mean is preserved)")
print("    - Std profile    : possibly over- or under-dispersed")
print("    - Covariance     : COLLAPSED — off-diagonal structure lost")
print("      because the average denoiser cannot reproduce the")
print("      noise-level-specific correlations in the residuals.")
print()

# ── 1. Instantiate blind model ────────────────────────────────────────────
torch.manual_seed(42)
model_blind     = DiffusionMLP(use_timestep_embedding=False)
lightning_blind = DiffusionLightning(
    model = model_blind,
    alpha = alpha,
    sigma = sigma,
    T     = T,
    lr    = LR_44,
)

n_full  = sum(p.numel() for p in lightning_full.parameters())
n_blind = sum(p.numel() for p in lightning_blind.parameters())
print(f"── Model comparison ─────────────────────────────────────────────────")
print(f"  Full model  params : {n_full:,}")
print(f"  Blind model params : {n_blind:,}")
print(f"  Difference         : {n_full - n_blind:,}  "
      f"(the TimestepEmbedding overhead)")

# ── 2. Train the blind model ──────────────────────────────────────────────
print(f"\nTraining blind model ({MAX_EPOCHS_44} epochs)...")

# Try WandB first, fall back to CSV
try:
    import wandb
    wandb_logger_blind = WandbLogger(
        project = PROJECT,
        name    = RUN_NAME_BLIND,
        log_model = False,
    )
    wandb_logger_blind.log_hyperparams({
        "use_timestep_embedding": False,
        "n_params": n_blind,
        "max_epochs": MAX_EPOCHS_44,
        "lr": LR_44,
    })
    logger_blind = wandb_logger_blind
except Exception:
    logger_blind = CSVLogger(".", name="blind_logs", version=0)

ckpt_cb_blind = ModelCheckpoint(
    monitor="val_loss", mode="min", save_top_k=1, verbose=False)
early_blind   = EarlyStopping(
    monitor="val_loss", patience=40, mode="min", verbose=True)

trainer_blind = pl.Trainer(
    max_epochs        = MAX_EPOCHS_44,
    logger            = logger_blind,
    callbacks         = [ckpt_cb_blind, early_blind],
    accelerator       = "auto",
    devices           = "auto",
    log_every_n_steps = 10,
    enable_progress_bar = True,
)
trainer_blind.fit(lightning_blind, train_loader, val_loader)
trainer_blind.save_checkpoint(CKPT_BLIND)
print(f"Blind model checkpoint saved to: {CKPT_BLIND}")

# ── 3. Extract loss history ───────────────────────────────────────────────
def get_loss_history(logger, trainer):
    """Extract train/val loss arrays from logger."""
    try:
        if hasattr(logger, "experiment") and hasattr(logger.experiment, "id"):
            # WandB
            import wandb
            api  = wandb.Api()
            run  = api.run(f"{logger.experiment.entity}/"
                           f"{PROJECT}/{logger.experiment.id}")
            hist = run.history(samples=MAX_EPOCHS_44)
            return (hist["train_loss"].dropna().values,
                    hist["val_loss"].dropna().values)
    except Exception:
        pass
    # CSV fallback
    for path in [
            "./blind_logs/version_0/metrics.csv",
            "./diffusion_logs/version_0/metrics.csv",
    ]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            tr = df[df["train_loss"].notna()]["train_loss"].values
            vl = df[df["val_loss"].notna()]["val_loss"].values
            return tr, vl
    return np.array([]), np.array([])

train_loss_blind, val_loss_blind = get_loss_history(logger_blind, trainer_blind)

# Also get full model losses if we have them
try:
    train_loss_full, val_loss_full = get_loss_history(wandb_logger, trainer)
except Exception:
    train_loss_full  = np.array([])
    val_loss_full    = np.array([])

# ── 4. Generate samples from blind model ─────────────────────────────────
print(f"\nGenerating {N_COMPARE} samples from blind model...")
torch.manual_seed(0)
samples_blind = sample(
    lightning_blind,
    batch_size = N_COMPARE,
    data_dim   = 15,
    device     = DEVICE,
).cpu().numpy()
print(f"  Generated: {samples_blind.shape}")

# Re-use training residuals from Task 43
# train_sub is already defined as (1000, 15) numpy array

# ── 5. Compute statistics for both models ────────────────────────────────
def compute_stats(arr):
    return {
        "mean": arr.mean(axis=0),
        "std" : arr.std(axis=0),
        "cov" : np.cov(arr.T),
        "corr": np.corrcoef(arr.T),
    }

stats_train = compute_stats(train_sub)
stats_full  = compute_stats(samples_full)   # from Task 43
stats_blind = compute_stats(samples_blind)

mean_mse_blind = float(np.mean(
    (stats_blind["mean"] - stats_train["mean"])**2))
std_ratio_full  = stats_full["std"].mean()  / stats_train["std"].mean()
std_ratio_blind = stats_blind["std"].mean() / stats_train["std"].mean()

offdiag_train = np.abs(stats_train["cov"] -
                        np.diag(np.diag(stats_train["cov"]))).mean()
offdiag_full  = np.abs(stats_full["cov"]  -
                        np.diag(np.diag(stats_full["cov"]))).mean()
offdiag_blind = np.abs(stats_blind["cov"] -
                        np.diag(np.diag(stats_blind["cov"]))).mean()

cov_diff_full  = np.abs(stats_full["cov"]  - stats_train["cov"]).mean()
cov_diff_blind = np.abs(stats_blind["cov"] - stats_train["cov"]).mean()

# KS tests
ks_full  = np.mean([stats.ks_2samp(train_sub[:,k],
                                    samples_full[:,k]).statistic
                    for k in range(15)])
ks_blind = np.mean([stats.ks_2samp(train_sub[:,k],
                                    samples_blind[:,k]).statistic
                    for k in range(15)])

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: Loss curves — full vs blind
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(1, 2, figsize=(14, 5))

for ax, scale in [(axes1[0], "linear"), (axes1[1], "log")]:
    plot_fn = ax.plot if scale == "linear" else ax.semilogy

    if len(train_loss_full) > 0:
        plot_fn(train_loss_full, color="tab:blue", linewidth=1.8,
                label="Full — train")
    if len(val_loss_full) > 0:
        plot_fn(val_loss_full, color="tab:blue", linewidth=1.8,
                linestyle="--", alpha=0.7, label="Full — val")
    if len(train_loss_blind) > 0:
        plot_fn(train_loss_blind, color="tab:red", linewidth=1.8,
                label="Blind — train")
    if len(val_loss_blind) > 0:
        plot_fn(val_loss_blind, color="tab:red", linewidth=1.8,
                linestyle="--", alpha=0.7, label="Blind — val")

    ax.set_xlabel("Epoch")
    ax.set_ylabel(f"MSE loss ({scale} scale)")
    ax.set_title(f"Training curves — full vs blind ({scale} scale)")
    ax.legend(fontsize=8)

fig1.suptitle(
    "Loss curves: full model (with t-emb, blue) vs blind model (no t-emb, red)\n"
    "If blind ≈ full: t-embedding not needed for this dataset.  "
    "If blind >> full: t-embedding earns its keep.",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: Covariance comparison — the headline result
# ══════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 3, figsize=(17, 5))

cov_absmax = max(np.abs(stats_train["cov"]).max(),
                 np.abs(stats_full["cov"]).max(),
                 np.abs(stats_blind["cov"]).max())
norm_cov   = mcolors.TwoSlopeNorm(
    vmin=-cov_absmax, vcenter=0, vmax=cov_absmax)

ticks      = np.arange(0, 15, 3)
tick_labels= [f"{LAT_CENTERS_43[i]:.0f}°" for i in ticks]

for ax, mat, title, extra in [
        (axes2[0], stats_train["cov"],
         "Training covariance\n(target)", ""),
        (axes2[1], stats_full["cov"],
         f"Full model (with t-emb)\nmean|diff|={cov_diff_full:.4f}", ""),
        (axes2[2], stats_blind["cov"],
         f"Blind model (no t-emb)\nmean|diff|={cov_diff_blind:.4f}", ""),
]:
    im = ax.imshow(mat, cmap="RdBu_r", norm=norm_cov, aspect="auto")
    ax.set_title(title, fontsize=9)
    ax.set_xticks(ticks); ax.set_xticklabels(tick_labels, fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(tick_labels, fontsize=7)
    ax.set_xlabel("Bin"); ax.set_ylabel("Bin")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig2.suptitle(
    "Covariance structure: training vs full model vs blind model\n"
    "HEADLINE RESULT — does the blind model's covariance match training, "
    "or has it collapsed?",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: Mean + std side-by-side — all three
# ══════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))

width = BIN_WIDTH_43 * 0.28

# Mean profiles
ax = axes3[0]
ax.bar(LAT_CENTERS_43 - width, stats_train["mean"], width=width,
       color="black",      alpha=0.75, label="Training")
ax.bar(LAT_CENTERS_43,         stats_full["mean"],  width=width,
       color="tab:blue",   alpha=0.75, label="Full model")
ax.bar(LAT_CENTERS_43 + width, stats_blind["mean"], width=width,
       color="tab:red",    alpha=0.75, label="Blind model")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Latitude (°)"); ax.set_ylabel("Mean residual density")
ax.set_title(f"Bin-wise mean\nMSE: full={mean_mse_full:.2e}  "
             f"blind={mean_mse_blind:.2e}")
ax.legend(fontsize=7.5)

# Std profiles
ax2 = axes3[1]
ax2.bar(LAT_CENTERS_43 - width, stats_train["std"], width=width,
        color="black",      alpha=0.75, label="Training")
ax2.bar(LAT_CENTERS_43,         stats_full["std"],  width=width,
        color="tab:blue",   alpha=0.75, label="Full model")
ax2.bar(LAT_CENTERS_43 + width, stats_blind["std"], width=width,
        color="tab:red",    alpha=0.75, label="Blind model")
ax2.set_xlabel("Latitude (°)"); ax2.set_ylabel("Std of residual density")
ax2.set_title(f"Bin-wise std\nstd ratio: full={std_ratio_full:.3f}  "
              f"blind={std_ratio_blind:.3f}")
ax2.legend(fontsize=7.5)

fig3.suptitle("Mean and std: training vs full model vs blind model", fontsize=9)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 4: Visual overlay — generated samples both models
# ══════════════════════════════════════════════════════════════════════════
fig4, axes4 = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, samples_plot, name, color in [
        (axes4[0], samples_full,  "Full model (with t-emb)", "tab:blue"),
        (axes4[1], samples_blind, "Blind model (no t-emb)",  "tab:red")]:

    idx_20 = rng_43.choice(N_COMPARE, size=20, replace=False)
    for i, idx in enumerate(idx_20):
        r = samples_plot[idx]
        ax.step(
            np.append(LAT_CENTERS_43 - 1.5, LAT_CENTERS_43[-1] + 1.5),
            np.append(r, r[-1]),
            color=color, linewidth=0.9, alpha=0.3, where="post",
            label="Generated" if i == 0 else "_"
        )

    # Also plot 20 training residuals faintly
    idx_tr = rng_43.choice(len(train_all), size=20, replace=False)
    for i, idx in enumerate(idx_tr):
        r = train_all[idx]
        ax.step(
            np.append(LAT_CENTERS_43 - 1.5, LAT_CENTERS_43[-1] + 1.5),
            np.append(r, r[-1]),
            color="black", linewidth=0.7, alpha=0.2, where="post",
            label="Training" if i == 0 else "_"
        )

    # Means
    ax.plot(LAT_CENTERS_43, samples_plot.mean(axis=0),
            color=color, linewidth=2.5, label="Generated mean")
    ax.plot(LAT_CENTERS_43, stats_train["mean"],
            color="black",  linewidth=2.5, linestyle="--",
            label="Training mean")
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xlabel("Latitude (°)")
    ax.set_ylabel("Residual density")
    ax.set_title(f"{name}\n"
                 f"MSE(means)={mean_mse_full if 'Full' in name else mean_mse_blind:.2e}  "
                 f"KS={ks_full if 'Full' in name else ks_blind:.4f}")
    ax.legend(fontsize=7.5, loc="upper right")
    ax.set_xlim(0, 45)

fig4.suptitle(
    "Visual overlay: 20 generated (colored) + 20 training (black faint)\n"
    "Thick lines = means.  "
    "Good model: colored cloud overlaps black cloud.",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Comparison table — the deliverable
# ══════════════════════════════════════════════════════════════════════════
final_train_full  = float(train_loss_full[-1])  if len(train_loss_full)  > 0 else float("nan")
final_val_full    = float(val_loss_full[-1])    if len(val_loss_full)    > 0 else float("nan")
final_train_blind = float(train_loss_blind[-1]) if len(train_loss_blind) > 0 else float("nan")
final_val_blind   = float(val_loss_blind[-1])   if len(val_loss_blind)   > 0 else float("nan")

cov_match_full  = "YES" if cov_diff_full  < cov_diff_blind * 0.9 else \
                  "TIED" if abs(cov_diff_full - cov_diff_blind) < 0.005 else "NO"
cov_match_blind = "YES" if cov_diff_blind < cov_diff_full  * 0.9 else \
                  "TIED" if abs(cov_diff_full - cov_diff_blind) < 0.005 else "NO"

table_data = {
    "Model"              : ["Training data", "Full (with t-emb)", "Blind (no t-emb)"],
    "Final train loss"   : ["—",
                             f"{final_train_full:.5f}",
                             f"{final_train_blind:.5f}"],
    "Final val loss"     : ["—",
                             f"{final_val_full:.5f}",
                             f"{final_val_blind:.5f}"],
    "Mean MSE vs train"  : ["0.00000000",
                             f"{mean_mse_full:.8f}",
                             f"{mean_mse_blind:.8f}"],
    "Std ratio vs train" : ["1.000",
                             f"{std_ratio_full:.3f}",
                             f"{std_ratio_blind:.3f}"],
    "Mean KS statistic"  : ["0.000",
                             f"{ks_full:.4f}",
                             f"{ks_blind:.4f}"],
    "Cov match training" : ["—", cov_match_full, cov_match_blind],
    "Off-diag cov diff"  : ["0.000000",
                             f"{cov_diff_full:.6f}",
                             f"{cov_diff_blind:.6f}"],
}
df_table = pd.DataFrame(table_data)

print("\n" + "=" * 90)
print("  ABLATION COMPARISON TABLE")
print("=" * 90)
print(df_table.to_string(index=False))
print("=" * 90)

# ── Verdict ───────────────────────────────────────────────────────────────
print("\n── Verdict ──────────────────────────────────────────────────────────")
loss_delta = final_val_full - final_val_blind  if not np.isnan(final_val_full) else 0
cov_delta  = cov_diff_full  - cov_diff_blind

if loss_delta < -0.005 and cov_delta < -0.005:
    verdict = ("FULL MODEL WINS — the timestep embedding earns its keep.\n"
               "  Both loss and covariance match are better with t-embedding.\n"
               "  Build Week 09 conditioning on top of the t-aware architecture.")
elif cov_delta < -0.005:
    verdict = ("FULL MODEL WINS ON COVARIANCE — t-embedding helps structure.\n"
               "  Loss difference is small but covariance match is better.\n"
               "  The embedding is doing real work on off-diagonal structure.")
elif abs(loss_delta) < 0.005 and abs(cov_delta) < 0.005:
    verdict = ("MODELS ARE TIED — t-embedding not contributing on this dataset.\n"
               "  Possible reasons: T=200 is more than needed for 15-dim data;\n"
               "  cosine schedule keeps signal alive → average denoiser works;\n"
               "  data lacks t-dependent structure at this scale.\n"
               "  Both findings are real and publishable.")
elif loss_delta > 0.005:
    verdict = ("BLIND MODEL WINS ON LOSS — CHECK FOR BUGS.\n"
               "  Re-run Task 39 t-sensitivity check on model_full.\n"
               "  Possible: TimestepEmbedding output is near-zero or constant.")
else:
    verdict = ("MIXED RESULT — loss similar, covariance differs.\n"
               f"  Full cov diff: {cov_diff_full:.4f}  "
               f"Blind cov diff: {cov_diff_blind:.4f}\n"
               "  Check Figure 2 (covariance heatmaps) to assess qualitatively.")

print(f"\n  {verdict}")

print(f"\n✓ Task 44 complete — ablation experiment finished")
print(f"  Checkpoints: {CKPT_BLIND}")
print(f"  Ready for Week 09: conditional diffusion model")


---
## Where Week 08 leaves us, and what Week 09 will need

By the end of Task 44 you have built every piece of the AI/ML pipeline a
PyTorch + Lightning project needs, instantiated for the unconditional
diffusion model:

- A **PyTorch Dataset** that wraps the Week 07 residual table loaded
  from `diffusion_windows.parquet` (Task 33).
- A set of **DataLoaders** for train / val / test using the parquet's
  `split` column with appropriate shuffling (Task 35).
- A **sinusoidal timestep embedding** that turns integer t into a dense
  vector (Task 36) and a **TimestepEmbedding module** that learns to
  project that vector into a useful representation (Task 37).
- A **DiffusionMLP** that takes (r_t, t) and predicts ε̂, with a flag for
  the architectural ablation (Task 38), sanity-tested for shape and
  t-sensitivity (Task 39).
- A **LightningModule** that stitches the model and the Week 07 schedule
  into a training loop (Task 40), trained end-to-end with WandB logging
  (Task 41).
- A **DDIM sampler** that runs the trained model in reverse to generate
  new residuals (Task 42).
- Two empirical results: the **distributional verification** that the
  trained model's samples match the training distribution (Task 43) and
  the **timestep-embedding ablation** that measures whether the
  diffusion-specific architecture earns its keep (Task 44).

What you have **not** yet built is conditioning: every sample drawn from
the trained model is from the *marginal* residual distribution, not from
a distribution targeted at a specific window. That is the Week 09 lift.

**What changes in Week 09.** The Dataset returns
`(r, amplitude, mu_universal)` tuples instead of just `r` (the parquet
already has those columns). The DiffusionMLP gains a second concatenation
input — the conditioning vector — alongside the timestep embedding. The
LightningModule's `training_step` passes the conditioning through. The
sampler accepts a per-sample conditioning vector and threads it through
each reverse step. None of the diffusion-specific machinery (the schedule,
the forward equation, the ε-prediction objective, the timestep embedding
itself) changes. The conditioning is an *extension* of the architecture
you have already built, not a replacement for it.

**What stays.** Every line of `training_step`. Every line of the sampler
loop. The LightningModule structure. The schedule buffers. The full
ablation infrastructure. Week 09 is a small refactor on a working
pipeline, not a rebuild.

**What we expect to see in Week 09.** With conditioning, the combined
classical + diffusion model (the official `ButterflAIModel.density(A, tau, ·)`
loaded as `classical` in this notebook's setup, plus a sampled residual
targeted at that window's amplitude and mu_universal) should outperform
the classical model alone on `compute_global_nll` for held-out test
cycles. That is the value-add metric the program has been pointing at
since Week 03. Whether the margin is large or small is itself a real
empirical question, and it is the question Week 09 finally lets you
answer.
